# v9_b Step 1: Candidate filtering for four derivatization–matrix datasets

This runs candidate-filter workflow for DnCl pellets, DnCl supernatant, DnHz pellets and DnHz supernatant. Input and output paths are relative to this project folder. The original filtering rules, exports, markdown explanations and heatmap code are preserved dataset by dataset.

The four full feature tables generate ten steroid–matrix result groups: Estradiol and Estrone for each DnCl matrix, and DHEA, Pregnenolone and Progesterone for each DnHz matrix.

In [ ]:
import os
import numpy as np
import pandas as pd
import scipy.stats as stats
import matplotlib.pyplot as plt
from matplotlib.colors import Normalize
import seaborn as sns
from pathlib import Path
from asari.tools.match_features import list_match_lcms_features

deu = DEUTERIUM_MASS_SHIFT = 1.006276  # 2.014101 - 1.007825

LOG2FC_THRESHOLD = 3.0
FDR_THRESHOLD = 0.05
MZ_PPM = 10
RT_TOLERANCE_SECONDS = 5
SIGNAL_THRESHOLD_RAW = 1_000_000.0
ENZYME_VS_WT_LOG2FC_THRESHOLD = 1.0
ENZYME_VS_WT_FDR_THRESHOLD = 0.05

PROJECT_DIR = './input/Feature_tables/'      # this is really input dir, not project dir
OUTPUT_DIR = 'results/step1_filtered_results/'
os.makedirs(OUTPUT_DIR, exist_ok=True)

In [ ]:

H_D_MATCH_EXPORT_COLUMNS = [
    'unlabeled_feature',
    'unlabeled_feature_mz',
    'unlabeled_feature_rtime',
    'enzyme',
    'unlabeled_treatment_mean_normalized_raw',
    'unlabeled_dmso_mean_normalized_raw',
    'unlabeled_log2FC',
    'unlabeled_fdr_q',
    'labeled_feature',
    'labeled_feature_mz',
    'labeled_feature_rtime',
    'deuterium_label',
    'labeled_treatment_mean_normalized_raw',
    'labeled_dmso_mean_normalized_raw',
    'labeled_log2FC',
    'labeled_fdr_q',
]

ENZYME_VS_WT_EXPORT_COLUMNS = [
    'enzyme',
    'unlabeled_feature',
    'enzyme_detected_count',
    'wt_detected_count',
    'enzyme_original_raw_rep1',
    'enzyme_original_raw_rep2',
    'enzyme_original_raw_rep3',
    'wt_original_raw_rep1',
    'wt_original_raw_rep2',
    'wt_original_raw_rep3',
    'enzyme_mean_normalized_raw',
    'wt_mean_normalized_raw',
    'enzyme_vs_wt_log2FC',
    'enzyme_vs_wt_fdr_q',
    'specificity_evidence',
    'pass_enzyme_vs_wt',
    'number_of_enzymes_passing',
    'specificity_class',
]

ENZYME_SPECIFIC_PAIR_EXPORT_COLUMNS = [
    'unlabeled_feature',
    'unlabeled_feature_mz',
    'unlabeled_feature_rtime',
    'enzyme',
    'labeled_feature',
    'labeled_feature_mz',
    'labeled_feature_rtime',
    'deuterium_label',
    'unlabeled_treatment_mean_normalized_raw',
    'labeled_treatment_mean_normalized_raw',
    'enzyme_mean_normalized_raw',
    'wt_mean_normalized_raw',
    'enzyme_vs_wt_log2FC',
    'enzyme_vs_wt_fdr_q',
    'specificity_evidence',
    'number_of_enzymes_passing',
    'specificity_class',
]

UNIQUE_ENZYME_SPECIFIC_FEATURE_EXPORT_COLUMNS = [
    'unlabeled_feature',
    'unlabeled_feature_mz',
    'unlabeled_feature_rtime',
    'enzyme',
    'unlabeled_treatment_mean_normalized_raw',
    'enzyme_mean_normalized_raw',
]

ordered_enzyme_names = [
    '1A1', '1A2', '2A6', '2B6', '2C19', '2C8',
    '2C9', '2D6', '2E1', '3A4', '3A5', 'WT',
]

def deuterium_search_range(max_observed_deuterium):
    return [
        (DEUTERIUM_MASS_SHIFT * d, f'D/H*{d}')
        for d in range(1, max_observed_deuterium + 1)
    ]
    
# D4-Estrone and D4-Estradiol can retain at most four deuterium atoms.
deu_search = deuterium_search_range(4)
# [(deu * d, f'D/H*{d}') for d in range(1, 5)]
deu_search


In [ ]:
def format_enzyme_label(value):
    '''Use standard CYP names in exported enzyme columns.'''
    label = str(value)
    if label == 'WT' or label.startswith('CYP'):
        return label
    return f'CYP{label}'

def export_selected_columns(table, columns, output_path):
    '''Write a compact, explicitly ordered TSV and fail on schema drift.'''
    missing_columns = [column for column in columns if column not in table.columns]
    if missing_columns:
        raise KeyError(f'Missing export columns: {missing_columns}')
    export_table = table.loc[:, columns].copy()
    if 'enzyme' in export_table.columns:
        export_table['enzyme'] = export_table['enzyme'].map(format_enzyme_label)
    ## output_path.parent.mkdir(parents=True, exist_ok=True)
    export_table.to_csv(output_path, sep='\t', index=False)


def samples_for_enzyme(sample_names, enzyme_name, expected_replicates=3):
    '''Select replicates by the enzyme token in the sample name, never by position.'''
    selected = sorted(
        name for name in sample_names
        if name.split('_', 1)[0] == enzyme_name
    )
    if len(selected) != expected_replicates:
        raise ValueError(
            f'{enzyme_name}: expected {expected_replicates} replicates, '
            f'found {len(selected)}: {selected}'
        )
    return selected


def normalize_top500_then_log2(raw_data, n_top=500):
    '''Select top features on raw means, normalize raw values, then log2(x+1).'''
    top_features = raw_data.mean(axis=1).nlargest(n_top).index
    raw_top_tic = raw_data.loc[top_features].sum(axis=0)
    if (raw_top_tic <= 0).any():
        raise ValueError('Top-feature TIC must be positive in every sample.')
    sample_scale = raw_top_tic / raw_top_tic.mean()
    normalized_raw = raw_data.div(sample_scale, axis=1)
    normalized_log2 = np.log2(normalized_raw + 1)
    return normalized_raw, normalized_log2, sample_scale


def plot_normalization_qc(raw_data, normalized_raw, substrate_name):
    '''Compare all-feature raw TIC before and after normalization on one y-scale.'''
    before_tic = raw_data.sum(axis=0).to_numpy()
    after_tic = normalized_raw.sum(axis=0).to_numpy()
    all_tic = np.concatenate([before_tic, after_tic])
    y_min, y_max = all_tic.min(), all_tic.max()
    y_padding = 0.05 * (y_max - y_min)

    fig, axes = plt.subplots(1, 2, figsize=(12, 3))
    axes[0].plot(before_tic, 'o', ms=3)
    axes[0].set_title(f'{substrate_name} set: raw total intensity')
    axes[1].plot(after_tic, 'o', ms=3)
    axes[1].set_title('After top-500 normalization: all-feature raw TIC')
    for axis in axes:
        axis.set_xlabel('Sample index')
        axis.set_ylim(y_min - y_padding, y_max + y_padding)
    fig.tight_layout()
    plt.show()

def bh_fdr(p_values):
    '''Benjamini-Hochberg correction; non-finite p-values receive q=1.'''
    p = np.asarray(p_values, dtype=float)
    q = np.ones(p.shape, dtype=float)
    valid = np.isfinite(p)
    pv = p[valid]
    if pv.size == 0:
        return q
    order = np.argsort(pv)
    ranked = pv[order] * pv.size / np.arange(1, pv.size + 1)
    adjusted = np.minimum.accumulate(ranked[::-1])[::-1]
    qv = np.empty_like(adjusted)
    qv[order] = np.clip(adjusted, 0, 1)
    q[valid] = qv
    return q


def welch_pvalues(treatment_log2, dmso_log2):
    '''Vectorized two-sided Welch t-test for every feature.'''
    n_t = treatment_log2.shape[1]
    n_d = dmso_log2.shape[1]
    mean_t = treatment_log2.mean(axis=1).to_numpy(dtype=float)
    mean_d = dmso_log2.mean(axis=1).to_numpy(dtype=float)
    var_t = treatment_log2.var(axis=1, ddof=1).to_numpy(dtype=float)
    var_d = dmso_log2.var(axis=1, ddof=1).to_numpy(dtype=float)

    term_t = var_t / n_t
    term_d = var_d / n_d
    se2 = term_t + term_d
    difference = mean_t - mean_d
    t_stat = np.divide(
        difference, np.sqrt(se2),
        out=np.zeros_like(difference), where=se2 > 0,
    )
    df_denominator = (term_t ** 2) / (n_t - 1) + (term_d ** 2) / (n_d - 1)
    degrees_freedom = np.divide(
        se2 ** 2, df_denominator,
        out=np.full_like(se2, np.inf), where=df_denominator > 0,
    )
    p_values = 2 * stats.t.sf(np.abs(t_stat), degrees_freedom)
    p_values[(se2 == 0) & (difference == 0)] = 1.0
    p_values[(se2 == 0) & (difference != 0)] = 0.0
    p_values[~np.isfinite(p_values)] = 1.0
    return p_values


def treatment_enrichment_metrics(
    original_raw, normalized_raw, normalized_log2,
    treatment_samples, dmso_samples,
    log2fc_threshold=LOG2FC_THRESHOLD,
    fdr_threshold=FDR_THRESHOLD,
):
    '''Calculate all criteria and the final pass flag for one 3-vs-36 contrast.'''
    detection_count = (original_raw[treatment_samples] > 0).sum(axis=1)
    treatment_mean = normalized_raw[treatment_samples].mean(axis=1)
    dmso_mean = normalized_raw[dmso_samples].mean(axis=1)
    log2fc = np.log2((treatment_mean + 1) / (dmso_mean + 1))
    p_values = welch_pvalues(
        normalized_log2[treatment_samples], normalized_log2[dmso_samples]
    )
    q_values = bh_fdr(p_values)

    metrics = pd.DataFrame({
        'detection_count': detection_count.astype(int),
        'treatment_mean_normalized_raw': treatment_mean,
        'dmso_mean_normalized_raw': dmso_mean,
        'log2FC': log2fc,
        'welch_p': p_values,
        'fdr_q': q_values,
    }, index=normalized_raw.index)
    metrics['pass'] = (
        (metrics['detection_count'] == len(treatment_samples))
        & (metrics['log2FC'] >= log2fc_threshold)
        & (metrics['treatment_mean_normalized_raw'] > metrics['dmso_mean_normalized_raw'])
        & (metrics['fdr_q'] < fdr_threshold)
    )
    return metrics

def screen_all_enzymes(
    original_raw, normalized_raw, normalized_log2,
    treatment_samples, dmso_samples, enzyme_names,
):
    '''Run one treatment route for all enzymes and retain passing metrics.'''
    selected_by_enzyme = {}
    for enzyme in enzyme_names:
        replicates = samples_for_enzyme(treatment_samples, enzyme)
        metrics = treatment_enrichment_metrics(
            original_raw, normalized_raw, normalized_log2,
            replicates, dmso_samples,
        )
        selected_by_enzyme[enzyme] = metrics.loc[metrics['pass']].copy()
    return selected_by_enzyme


def summarize_screen_counts(unlabeled_selected, labeled_selected, enzyme_names):
    '''Summarize passing unlabeled and labeled features before matching.'''
    counts = pd.DataFrame({
        'unlabeled_pass': {
            enzyme: len(unlabeled_selected[enzyme]) for enzyme in enzyme_names
        },
        'labeled_pass': {
            enzyme: len(labeled_selected[enzyme]) for enzyme in enzyme_names
        },
    })
    counts.loc['unique_features'] = [
        len(set().union(*(set(values.index) for values in unlabeled_selected.values()))),
        len(set().union(*(set(values.index) for values in labeled_selected.values()))),
    ]
    return counts


def deu_search_list(
    selected_feature_ids, feature_dict, full_feature_list,
    deu_search, mz_ppm=MZ_PPM, rt_tolerance=RT_TOLERANCE_SECONDS,
):
    '''Search D1-D4 mass shifts around unlabeled features.'''
    selected_feature_ids = list(selected_feature_ids)
    if not selected_feature_ids:
        return {}
    calculated_deu_list = []
    for feature_id in selected_feature_ids:
        for mass_shift, label in deu_search:
            calculated_deu_list.append({
                'id': f'{feature_id}, {label}',
                'mz': feature_dict[feature_id]['mz'] + mass_shift,
                'rtime': feature_dict[feature_id]['rtime'],
            })
    return list_match_lcms_features(
        calculated_deu_list, full_feature_list,
        mz_ppm=mz_ppm, rt_tolerance=rt_tolerance,
    )


def match_selected_routes(
    unlabeled_selected, labeled_selected, enzyme_names,
    feature_dict, feature_list, deu_search=deu_search
):
    '''Mass/RT match independently screened features within the same enzyme.'''
    matched_by_enzyme = {}
    for enzyme in enzyme_names:
        candidates = deu_search_list(
            unlabeled_selected[enzyme].index,
            feature_dict, feature_list, deu_search,
        )
        allowed_labeled = set(labeled_selected[enzyme].index)
        matched_by_enzyme[enzyme] = {
            key: [feature_id for feature_id in values if feature_id in allowed_labeled]
            for key, values in candidates.items()
            if any(feature_id in allowed_labeled for feature_id in values)
        }
    return matched_by_enzyme


def matches_to_table(matched_by_enzyme, unlabeled_selected, labeled_selected, feature_dict):
    '''Create a result table containing statistics for both members of every pair.'''
    rows = []
    for enzyme, matches in matched_by_enzyme.items():
        for key, labeled_ids in matches.items():
            unlabeled_feature_id, deuterium_label = key.split(', ')
            unlabeled_feature = unlabeled_selected[enzyme].loc[unlabeled_feature_id]
            for labeled_id in labeled_ids:
                labeled = labeled_selected[enzyme].loc[labeled_id]
                rows.append({
                    'unlabeled_feature': unlabeled_feature_id,
                    'unlabeled_feature_mz': feature_dict[unlabeled_feature_id]['mz'],
                    'unlabeled_feature_rtime': feature_dict[unlabeled_feature_id]['rtime'],
                    'enzyme': enzyme,
                    'unlabeled_detection_count': int(unlabeled_feature['detection_count']),
                    'unlabeled_treatment_mean_normalized_raw': unlabeled_feature['treatment_mean_normalized_raw'],
                    'unlabeled_dmso_mean_normalized_raw': unlabeled_feature['dmso_mean_normalized_raw'],
                    'unlabeled_log2FC': unlabeled_feature['log2FC'],
                    'unlabeled_welch_p': unlabeled_feature['welch_p'],
                    'unlabeled_fdr_q': unlabeled_feature['fdr_q'],
                    'labeled_feature': labeled_id,
                    'labeled_feature_mz': feature_dict[labeled_id]['mz'],
                    'labeled_feature_rtime': feature_dict[labeled_id]['rtime'],
                    'deuterium_label': deuterium_label,
                    'labeled_detection_count': int(labeled['detection_count']),
                    'labeled_treatment_mean_normalized_raw': labeled['treatment_mean_normalized_raw'],
                    'labeled_dmso_mean_normalized_raw': labeled['dmso_mean_normalized_raw'],
                    'labeled_log2FC': labeled['log2FC'],
                    'labeled_welch_p': labeled['welch_p'],
                    'labeled_fdr_q': labeled['fdr_q'],
                })
    return pd.DataFrame(rows)


def enzyme_vs_wt_candidate_statistics(
    result_table, original_raw, normalized_raw, normalized_log2,
    unlabeled_samples, enzyme_names,
    signal_threshold_raw=SIGNAL_THRESHOLD_RAW,
    log2fc_threshold=ENZYME_VS_WT_LOG2FC_THRESHOLD,
    fdr_threshold=ENZYME_VS_WT_FDR_THRESHOLD,
):
    '''Apply only WT 0/3 detection or WT 3/3 quantitative routes.'''
    wt_samples = samples_for_enzyme(unlabeled_samples, 'WT')
    result_rows = []

    for enzyme in [name for name in enzyme_names if name != 'WT']:
        candidates = sorted(
            result_table.loc[
                result_table['enzyme'].eq(enzyme), 'unlabeled_feature'
            ].unique()
        )
        if not candidates:
            continue

        enzyme_samples = samples_for_enzyme(unlabeled_samples, enzyme)
        enzyme_raw = original_raw.loc[candidates, enzyme_samples]
        wt_raw = original_raw.loc[candidates, wt_samples]

        enzyme_detected_count = (enzyme_raw > 0).sum(axis=1)
        wt_detected_count = (wt_raw > 0).sum(axis=1)
        enzyme_mean_raw = enzyme_raw.mean(axis=1)
        wt_mean_raw = wt_raw.mean(axis=1)
        enzyme_min_raw = enzyme_raw.min(axis=1)
        enzyme_max_raw = enzyme_raw.max(axis=1)
        wt_min_raw = wt_raw.min(axis=1)
        wt_max_raw = wt_raw.max(axis=1)
        enzyme_all_raw_ge_threshold = enzyme_raw.ge(
            signal_threshold_raw
        ).all(axis=1)

        enzyme_normalized = normalized_raw.loc[candidates, enzyme_samples]
        wt_normalized = normalized_raw.loc[candidates, wt_samples]
        enzyme_mean = enzyme_normalized.mean(axis=1)
        wt_mean = wt_normalized.mean(axis=1)
        log2fc = np.log2((enzyme_mean + 1) / (wt_mean + 1))

        quantitative_test_mask = (
            enzyme_detected_count.eq(3)
            & wt_detected_count.eq(3)
            & enzyme_all_raw_ge_threshold
        )
        p_values = np.full(len(candidates), np.nan, dtype=float)
        q_values = np.full(len(candidates), np.nan, dtype=float)
        if quantitative_test_mask.any():
            quantitative_candidates = list(
                quantitative_test_mask.index[quantitative_test_mask]
            )
            quantitative_p = welch_pvalues(
                normalized_log2.loc[quantitative_candidates, enzyme_samples],
                normalized_log2.loc[quantitative_candidates, wt_samples],
            )
            quantitative_q = bh_fdr(quantitative_p)
            positions = quantitative_test_mask.to_numpy(dtype=bool)
            p_values[positions] = quantitative_p
            q_values[positions] = quantitative_q

        stats_table = pd.DataFrame({
            'enzyme': enzyme,
            'unlabeled_feature': candidates,
            'signal_threshold_raw': signal_threshold_raw,
            'enzyme_detected_count': enzyme_detected_count.to_numpy(dtype=int),
            'wt_detected_count': wt_detected_count.to_numpy(dtype=int),
            'enzyme_mean_original_raw': enzyme_mean_raw.to_numpy(dtype=float),
            'wt_mean_original_raw': wt_mean_raw.to_numpy(dtype=float),
            'enzyme_min_original_raw': enzyme_min_raw.to_numpy(dtype=float),
            'enzyme_max_original_raw': enzyme_max_raw.to_numpy(dtype=float),
            'wt_min_original_raw': wt_min_raw.to_numpy(dtype=float),
            'wt_max_original_raw': wt_max_raw.to_numpy(dtype=float),
            'enzyme_all_raw_ge_1e6': (
                enzyme_all_raw_ge_threshold.to_numpy(dtype=bool)
            ),
            'enzyme_mean_normalized_raw': enzyme_mean.to_numpy(dtype=float),
            'wt_mean_normalized_raw': wt_mean.to_numpy(dtype=float),
            'enzyme_vs_wt_log2FC': log2fc.to_numpy(dtype=float),
            'enzyme_vs_wt_welch_p': p_values,
            'enzyme_vs_wt_fdr_q': q_values,
        })
        for replicate_number, sample_name in enumerate(enzyme_samples, start=1):
            stats_table[f'enzyme_original_raw_rep{replicate_number}'] = (
                enzyme_raw[sample_name].to_numpy(dtype=float)
            )
        for replicate_number, sample_name in enumerate(wt_samples, start=1):
            stats_table[f'wt_original_raw_rep{replicate_number}'] = (
                wt_raw[sample_name].to_numpy(dtype=float)
            )

        enzyme_three = stats_table['enzyme_detected_count'].eq(3)
        wt_zero = stats_table['wt_detected_count'].eq(0)
        wt_sparse = stats_table['wt_detected_count'].isin([1, 2])
        wt_three = stats_table['wt_detected_count'].eq(3)
        enzyme_above_threshold = stats_table['enzyme_all_raw_ge_1e6']

        detection_route = enzyme_three & wt_zero & enzyme_above_threshold
        quantitative_ready = enzyme_three & wt_three & enzyme_above_threshold
        quantitative_route = (
            quantitative_ready
            & stats_table['enzyme_mean_normalized_raw'].gt(
                stats_table['wt_mean_normalized_raw']
            )
            & stats_table['enzyme_vs_wt_log2FC'].ge(log2fc_threshold)
            & stats_table['enzyme_vs_wt_fdr_q'].lt(fdr_threshold)
        )

        stats_table['specificity_evidence'] = np.select(
            [
                ~enzyme_three,
                detection_route,
                wt_zero,
                wt_sparse,
                quantitative_ready,
                wt_three,
            ],
            [
                'enzyme_not_detected_3_of_3',
                'WT_0_of_3_detection_route',
                'WT_0_of_3_below_enzyme_threshold',
                'ambiguous_sparse_WT',
                'quantitative_Welch_test',
                'enzyme_3_of_3_below_raw_threshold',
            ],
            default='unexpected_detection_pattern',
        )
        stats_table['pass_enzyme_vs_wt'] = (
            detection_route | quantitative_route
        )
        result_rows.append(stats_table)

    if not result_rows:
        return pd.DataFrame()

    statistics_table = pd.concat(result_rows, ignore_index=True)
    passing_enzyme_count = (
        statistics_table.loc[statistics_table['pass_enzyme_vs_wt']]
        .groupby('unlabeled_feature')['enzyme'].nunique()
    )
    statistics_table['number_of_enzymes_passing'] = (
        statistics_table['unlabeled_feature']
        .map(passing_enzyme_count).fillna(0).astype(int)
    )
    statistics_table['specificity_class'] = np.select(
        [
            statistics_table['pass_enzyme_vs_wt']
            & statistics_table['number_of_enzymes_passing'].eq(1),
            statistics_table['pass_enzyme_vs_wt']
            & statistics_table['number_of_enzymes_passing'].gt(1),
        ],
        ['single_enzyme', 'shared_across_enzymes'],
        default='not_specific_vs_WT',
    )
    return statistics_table


def retain_specific_isotope_pairs(result_table, statistics_table):
    '''Merge WT statistics into isotope matches and retain passing rows.'''
    non_wt_pairs = result_table.loc[~result_table['enzyme'].eq('WT')].copy()
    merged = non_wt_pairs.merge(
        statistics_table,
        on=['enzyme', 'unlabeled_feature'],
        how='left',
        validate='many_to_one',
    )
    return merged.loc[merged['pass_enzyme_vs_wt']].copy()


def specificity_summary(statistics_table):
    '''Count the two passing routes and excluded sparse-WT cases by enzyme.'''
    summary_rows = []
    for enzyme, group in statistics_table.groupby('enzyme', sort=False):
        passed = group.loc[group['pass_enzyme_vs_wt']]
        summary_rows.append({
            'enzyme': enzyme,
            'tested_unlabeled_features': len(group),
            'passed_unlabeled_features': len(passed),
            'WT_0_of_3_route': (
                passed['specificity_evidence']
                .eq('WT_0_of_3_detection_route').sum()
            ),
            'quantitative_Welch_route': (
                passed['specificity_evidence']
                .eq('quantitative_Welch_test').sum()
            ),
            'ambiguous_sparse_WT': (
                group['specificity_evidence'].eq('ambiguous_sparse_WT').sum()
            ),
            'enzyme_3_of_3_below_raw_threshold': (
                group['specificity_evidence']
                .eq('enzyme_3_of_3_below_raw_threshold').sum()
            ),
            'single_enzyme_features': (
                passed['specificity_class'].eq('single_enzyme').sum()
            ),
            'shared_features': (
                passed['specificity_class'].eq('shared_across_enzymes').sum()
            ),
        })
    return pd.DataFrame(summary_rows).set_index('enzyme')


def plot_v6_candidate_features_heatmap(
    normalized_log2, unlabeled_samples, specificity_statistics,
    enzyme_names, steroid_name, fraction_name, figure_path, matrix_path,
):
    """Plot all WT-passing candidates using the final v9 heatmap layout."""
    passed = specificity_statistics.loc[
        specificity_statistics['pass_enzyme_vs_wt']
    ].copy()
    feature_ids = sorted(passed['unlabeled_feature'].unique())
    if not feature_ids:
        raise ValueError(f'No v6 WT-passing features available for {steroid_name}.')

    group_mean_log2 = pd.DataFrame({
        enzyme: normalized_log2.loc[
            feature_ids, samples_for_enzyme(unlabeled_samples, enzyme)
        ].mean(axis=1)
        for enzyme in enzyme_names
    })
    row_mean = group_mean_log2.mean(axis=1)
    row_sd = group_mean_log2.std(axis=1, ddof=0).replace(0, np.nan)
    row_zscore = (
        group_mean_log2.sub(row_mean, axis=0)
        .div(row_sd, axis=0)
        .fillna(0)
    )
    passing_enzyme = passed.groupby('unlabeled_feature')['enzyme'].agg(
        lambda values: values.iloc[0] if values.nunique() == 1 else 'Shared'
    ).reindex(row_zscore.index)

    figure_width = 12
    figure_height = 2.8 + 0.075 * len(row_zscore)
    sns.set_theme(style='white', font_scale=0.9)
    grid = sns.clustermap(
        row_zscore,
        row_cluster=len(row_zscore) > 1,
        col_cluster=False,
        cmap='vlag', center=0, vmin=-2.5, vmax=2.5,
        xticklabels=True, yticklabels=False,
        figsize=(figure_width, figure_height),
        dendrogram_ratio=(0.12, 0.08),
        cbar_pos=None,
    )
    grid.ax_heatmap.set_xticklabels(
        grid.ax_heatmap.get_xticklabels(), rotation=45, ha='right'
    )
    grid.ax_heatmap.set_xlabel(
        'Unlabeled substrate group (mean of 3 replicates)'
    )
    grid.ax_heatmap.set_ylabel('')
    grid.fig.suptitle(
        f'{steroid_name}_candidate_features ({fraction_name})',
        fontsize=15, y=0.985,
    )
    colorbar_ax = grid.fig.add_axes([0.03, 0.94, 0.12, 0.015])
    colorbar = grid.fig.colorbar(
        plt.cm.ScalarMappable(
            norm=Normalize(vmin=-2.5, vmax=2.5),
            cmap=sns.color_palette('vlag', as_cmap=True),
        ),
        cax=colorbar_ax, orientation='horizontal',
    )
    colorbar.ax.tick_params(labelsize=8)
    grid.fig.patch.set_facecolor('white')
    for axis in grid.fig.axes:
        axis.set_facecolor('white')
    grid.fig.savefig(
        figure_path, dpi=300, facecolor='white', transparent=False,
    )

    if grid.dendrogram_row is not None:
        ordered_zscore = row_zscore.iloc[
            grid.dendrogram_row.reordered_ind
        ].copy()
    else:
        ordered_zscore = row_zscore.copy()
    ordered_zscore.insert(
        0, 'passing_enzyme', passing_enzyme.reindex(ordered_zscore.index)
    )
    # matrix_path.parent.mkdir(parents=True, exist_ok=True)
    ordered_zscore.to_csv(matrix_path, sep='\t')
    plt.close(grid.fig)
    print(f'{steroid_name} v6 candidate features: {len(row_zscore):,}')
    print(figure_path)
    print(matrix_path)


# v9_b Step 1: Dataset: DnCl pellets

P450 DnCl pellets candidate screening (v6; two-route enzyme-vs-WT; 10 ppm, minimum peak height 1e6; 07252026)

Input: `DnCl_pellets_10ppm_min_peak_height_1e6_full_Feature_table.tsv`

This notebook replaces the previous z-score / DMSO-log2-threshold route.

For each enzyme, the three unlabeled replicates and the three D4 replicates are screened **independently**, each against all 36 DMSO samples. A feature passes a treatment screen only when:

1. raw intensity is greater than zero in all three treatment replicates;
2. `log2FC = log2((mean normalized raw treatment + 1) / (mean normalized raw DMSO + 1)) >= 3`;
3. normalized-raw treatment mean is greater than the DMSO mean;
4. two-sided Welch t-test on normalized `log2(x+1)` values has BH-FDR `q < 0.05`.

BH correction is performed separately for every enzyme × treatment type across all tested features. After both screens, only same-enzyme unlabeled/labeled feature pairs matching D1-D4, 10 ppm, and 5 seconds are retained.

In [ ]:
INPUT_FILE = PROJECT_DIR + 'DnCl_pellets_10ppm_min_peak_height_1e6_full_Feature_table.tsv'

In [ ]:
data_pellet_DnCl = pd.read_csv(INPUT_FILE, sep='\t', index_col=0)
print(f'Feature table shape: {data_pellet_DnCl.shape}')
data_pellet_DnCl[['mz', 'rtime']].head()

In [ ]:
DMSO_samples = sorted(x for x in data_pellet_DnCl.columns if '_DMSO_' in x)
Estrone_samples = sorted(x for x in data_pellet_DnCl.columns if '_Estrone_' in x)
D4_Estrone_samples = sorted(x for x in data_pellet_DnCl.columns if '_D4-Estrone_' in x)
Estradiol_samples = sorted(x for x in data_pellet_DnCl.columns if '_Estradiol_' in x)
D4_Estradiol_samples = sorted(x for x in data_pellet_DnCl.columns if '_D4-Estradiol_' in x)

expected_counts = {
    'DMSO': 36,
    'Estrone': 36,
    'D4-Estrone': 36,
    'Estradiol': 36,
    'D4-Estradiol': 36,
}
observed_counts = {
    'DMSO': len(DMSO_samples),
    'Estrone': len(Estrone_samples),
    'D4-Estrone': len(D4_Estrone_samples),
    'Estradiol': len(Estradiol_samples),
    'D4-Estradiol': len(D4_Estradiol_samples),
}
if observed_counts != expected_counts:
    raise ValueError(f'Unexpected sample counts: {observed_counts}')

for treatment_samples in [
    Estrone_samples, D4_Estrone_samples,
    Estradiol_samples, D4_Estradiol_samples,
]:
    for enzyme in ordered_enzyme_names:
        samples_for_enzyme(treatment_samples, enzyme)

observed_counts

In [ ]:
feature_dict = data_pellet_DnCl[['mz', 'rtime']].to_dict('index')
feature_list = [
    {'id': feature_id, 'mz': values['mz'], 'rtime': values['rtime']}
    for feature_id, values in feature_dict.items()
]
print(f'{len(feature_list):,} LC-MS features available for D1-D4 matching.')

### Estrone

In [ ]:
estrone_all_samples = DMSO_samples + Estrone_samples + D4_Estrone_samples
normalized_raw, normalized_log2, normalization_scale = normalize_top500_then_log2(
    data_pellet_DnCl[estrone_all_samples]
)

plot_normalization_qc(
    data_pellet_DnCl[estrone_all_samples], normalized_raw,
    'Estrone',
)
normalization_scale.describe()

In [ ]:
estrone_unlabeled_selected = screen_all_enzymes(
    data_pellet_DnCl, normalized_raw, normalized_log2,
    Estrone_samples, DMSO_samples, ordered_enzyme_names,
)
estrone_labeled_selected = screen_all_enzymes(
    data_pellet_DnCl, normalized_raw, normalized_log2,
    D4_Estrone_samples, DMSO_samples, ordered_enzyme_names,
)

In [ ]:
estrone_screen_counts = summarize_screen_counts(
    estrone_unlabeled_selected, estrone_labeled_selected, ordered_enzyme_names,
)
estrone_screen_counts

In [ ]:
estrone_matches = match_selected_routes(
    estrone_unlabeled_selected, estrone_labeled_selected,
    ordered_enzyme_names, feature_dict, feature_list,
)
estrone_results = matches_to_table(
    estrone_matches, estrone_unlabeled_selected,
    estrone_labeled_selected, feature_dict,
)
estrone_output = OUTPUT_DIR + "01_v7_estrone_DnCl_pellets_treatment_enriched_deuterium_matches.tsv"
export_selected_columns(
    estrone_results, H_D_MATCH_EXPORT_COLUMNS, estrone_output
)

print(f'Estrone matched rows: {len(estrone_results):,}')
print(f'Estrone unique unlabeled features: {estrone_results.unlabeled_feature.nunique() if not estrone_results.empty else 0:,}')
print(f'Estrone unique labeled features: {estrone_results.labeled_feature.nunique() if not estrone_results.empty else 0:,}')
print(estrone_output)

### Estradiol

In [ ]:
# Reuse these variable names so the large Estrone matrices can be released.
estradiol_all_samples = DMSO_samples + Estradiol_samples + D4_Estradiol_samples
normalized_raw, normalized_log2, normalization_scale = normalize_top500_then_log2(
    data_pellet_DnCl[estradiol_all_samples]
)

plot_normalization_qc(
    data_pellet_DnCl[estradiol_all_samples], normalized_raw,
    'Estradiol',
)
normalization_scale.describe()

In [ ]:
estradiol_unlabeled_selected = screen_all_enzymes(
    data_pellet_DnCl, normalized_raw, normalized_log2,
    Estradiol_samples, DMSO_samples, ordered_enzyme_names,
)
estradiol_labeled_selected = screen_all_enzymes(
    data_pellet_DnCl, normalized_raw, normalized_log2,
    D4_Estradiol_samples, DMSO_samples, ordered_enzyme_names,
)

In [ ]:
estradiol_screen_counts = summarize_screen_counts(
    estradiol_unlabeled_selected, estradiol_labeled_selected, ordered_enzyme_names,
)
estradiol_screen_counts

In [ ]:
estradiol_matches = match_selected_routes(
    estradiol_unlabeled_selected, estradiol_labeled_selected,
    ordered_enzyme_names, feature_dict, feature_list,
)
estradiol_results = matches_to_table(
    estradiol_matches, estradiol_unlabeled_selected,
    estradiol_labeled_selected, feature_dict,
)
estradiol_output = OUTPUT_DIR + "01_v7_estradiol_DnCl_pellets_treatment_enriched_deuterium_matches.tsv"

export_selected_columns(
    estradiol_results, H_D_MATCH_EXPORT_COLUMNS, estradiol_output
)

print(f'Estradiol matched rows: {len(estradiol_results):,}')
print(f'Estradiol unique unlabeled features: {estradiol_results.unlabeled_feature.nunique() if not estradiol_results.empty else 0:,}')
print(f'Estradiol unique labeled features: {estradiol_results.labeled_feature.nunique() if not estradiol_results.empty else 0:,}')
print(estradiol_output)

## Enzyme-versus-WT specificity screen

Only isotope-confirmed unlabeled feature candidates enter this stage. Detection
and the fixed 1e6 signal threshold use original raw intensities. Fold change
uses top-500-normalized raw means, and Welch testing uses normalized
`log2(x+1)` values.

All passing routes require enzyme detection in all three replicates:

1. **WT 0/3:** no t-test; require all three enzyme original raw intensities
   >= 1e6.
2. **WT 3/3:** require all three enzyme original raw intensities >= 1e6.
   WT has no raw-intensity threshold beyond 3/3 detection. Then require enzyme
   normalized-raw mean > WT normalized-raw mean, log2FC >= 1, and two-sided
   Welch BH-FDR q < 0.05.

WT 1/3 and WT 2/3 cases are labeled `ambiguous_sparse_WT` and cannot pass
enzyme specificity.


## Estrone: isotope-confirmed unlabeled features versus WT

In [ ]:
normalized_raw, normalized_log2, _ = normalize_top500_then_log2(
    data_pellet_DnCl[estrone_all_samples]
)
estrone_specificity_stats = enzyme_vs_wt_candidate_statistics(
    estrone_results, data_pellet_DnCl, normalized_raw, normalized_log2,
    Estrone_samples, ordered_enzyme_names,
)
estrone_specific_pairs = retain_specific_isotope_pairs(
    estrone_results, estrone_specificity_stats
)
estrone_specificity_summary = specificity_summary(estrone_specificity_stats)
estrone_specificity_summary

In [ ]:
estrone_stats_output = OUTPUT_DIR + "02_v7_estrone_DnCl_pellets_unlabeled_features_enzyme_vs_WT_statistics.tsv"
estrone_specific_output = OUTPUT_DIR + "03_v7_estrone_DnCl_pellets_enzyme_specific_isotope_pairs.tsv"
export_selected_columns(
    estrone_specificity_stats, ENZYME_VS_WT_EXPORT_COLUMNS,
    estrone_stats_output,
)
export_selected_columns(
    estrone_specific_pairs, ENZYME_SPECIFIC_PAIR_EXPORT_COLUMNS,
    estrone_specific_output,
)

estrone_unique_unlabeled_features = (
    estrone_specific_pairs.loc[
        :, UNIQUE_ENZYME_SPECIFIC_FEATURE_EXPORT_COLUMNS
    ]
    .drop_duplicates()
    .sort_values(['enzyme', 'unlabeled_feature'], kind='stable')
    .reset_index(drop=True)
)
estrone_unique_unlabeled_output = OUTPUT_DIR + "04_v7_estrone_DnCl_pellets_unique_enzyme_specific_unlabeled_features.tsv"

export_selected_columns(
    estrone_unique_unlabeled_features,
    UNIQUE_ENZYME_SPECIFIC_FEATURE_EXPORT_COLUMNS,
    estrone_unique_unlabeled_output,
)
print(f'Estrone tested enzyme-unlabeled-feature combinations: {len(estrone_specificity_stats):,}')
print(f'Estrone passing enzyme-unlabeled-feature combinations: {estrone_specificity_stats.pass_enzyme_vs_wt.sum():,}')
print(f'Estrone passing isotope-pair rows: {len(estrone_specific_pairs):,}')
print(estrone_stats_output)
print(estrone_specific_output)
print(
    f'Estrone unique enzyme-feature rows: '
    f'{len(estrone_unique_unlabeled_features):,}'
)
print(estrone_unique_unlabeled_output)

## Estradiol: isotope-confirmed unlabeled features versus WT

In [ ]:
normalized_raw, normalized_log2, _ = normalize_top500_then_log2(
    data_pellet_DnCl[estradiol_all_samples]
)
estradiol_specificity_stats = enzyme_vs_wt_candidate_statistics(
    estradiol_results, data_pellet_DnCl, normalized_raw, normalized_log2,
    Estradiol_samples, ordered_enzyme_names,
)
estradiol_specific_pairs = retain_specific_isotope_pairs(
    estradiol_results, estradiol_specificity_stats
)
estradiol_specificity_summary = specificity_summary(estradiol_specificity_stats)
estradiol_specificity_summary

In [ ]:
estradiol_stats_output = OUTPUT_DIR + "02_v7_estradiol_DnCl_pellets_unlabeled_features_enzyme_vs_WT_statistics.tsv"
estradiol_specific_output = OUTPUT_DIR + "03_v7_estradiol_DnCl_pellets_enzyme_specific_isotope_pairs.tsv"
export_selected_columns(
    estradiol_specificity_stats, ENZYME_VS_WT_EXPORT_COLUMNS,
    estradiol_stats_output,
)
export_selected_columns(
    estradiol_specific_pairs, ENZYME_SPECIFIC_PAIR_EXPORT_COLUMNS,
    estradiol_specific_output,
)

estradiol_unique_unlabeled_features = (
    estradiol_specific_pairs.loc[
        :, UNIQUE_ENZYME_SPECIFIC_FEATURE_EXPORT_COLUMNS
    ]
    .drop_duplicates()
    .sort_values(['enzyme', 'unlabeled_feature'], kind='stable')
    .reset_index(drop=True)
)
estradiol_unique_unlabeled_output = OUTPUT_DIR + "04_v7_estradiol_DnCl_pellets_unique_enzyme_specific_unlabeled_features.tsv"
export_selected_columns(
    estradiol_unique_unlabeled_features,
    UNIQUE_ENZYME_SPECIFIC_FEATURE_EXPORT_COLUMNS,
    estradiol_unique_unlabeled_output,
)
print(f'Estradiol tested enzyme-unlabeled-feature combinations: {len(estradiol_specificity_stats):,}')
print(f'Estradiol passing enzyme-unlabeled-feature combinations: {estradiol_specificity_stats.pass_enzyme_vs_wt.sum():,}')
print(f'Estradiol passing isotope-pair rows: {len(estradiol_specific_pairs):,}')
print(estradiol_stats_output)
print(estradiol_specific_output)
print(
    f'Estradiol unique enzyme-feature rows: '
    f'{len(estradiol_unique_unlabeled_features):,}'
)
print(estradiol_unique_unlabeled_output)

## Make candidate feature heatmaps

Each heatmap contains all v6 WT-passing unlabeled features, including single-enzyme and shared candidates. The three normalized `log2(x+1)` replicates are averaged within each of the 11 enzymes and WT, followed by a row z-score across the 12 group means. The v9 final heatmap layout is used for direct visual comparison; this visualization does not change candidate selection or statistical testing.


## Estrone candidate features heatmap


In [ ]:
_, normalized_log2, _ = normalize_top500_then_log2(
    data_pellet_DnCl[estrone_all_samples]
)
plot_v6_candidate_features_heatmap(
    normalized_log2,
    Estrone_samples,
    estrone_specificity_stats,
    ordered_enzyme_names,
    'Estrone',
    'pellets',
    OUTPUT_DIR + "v7_Estrone_candidate_features_pellets_row_zscore_heatmap.pdf",
    OUTPUT_DIR + "05_v7_estrone_DnCl_pellets_all_WT_passing_unlabeled_features_row_zscore_matrix.tsv",
)


## Estradiol candidate features heatmap


In [ ]:
_, normalized_log2, _ = normalize_top500_then_log2(
    data_pellet_DnCl[estradiol_all_samples]
)
plot_v6_candidate_features_heatmap(
    normalized_log2,
    Estradiol_samples,
    estradiol_specificity_stats,
    ordered_enzyme_names,
    'Estradiol',
    'pellets',
    OUTPUT_DIR + "v7_Estradiol_candidate_features_pellets_row_zscore_heatmap.pdf",
    OUTPUT_DIR + "05_v7_estradiol_DnCl_pellets_all_WT_passing_unlabeled_features_row_zscore_matrix.tsv"
)


**Summary**

Stage 1 is unchanged from v5: independently enriched unlabeled features and
labeled features are matched within the same enzyme at D1-D4, 10 ppm, and
5 seconds.

The v6 enzyme-versus-WT stage has only two passing routes. WT 0/3 uses the
original-raw detection-limit rule without a t-test. WT 3/3 uses normalized-raw
direction/log2FC and a Welch test on normalized `log2(x+1)` values with BH-FDR.
WT 1/3 and WT 2/3 are always retained as `ambiguous_sparse_WT` diagnostics and
never pass specificity.

Passing unlabeled features are merged back to isotope matches. The candidate overview heatmap contains all v6 WT-passing unlabeled features and uses the final v9 heatmap layout. Heatmap row z-scores are visualization-only and do not affect selection.


# v9_b Step 1: Dataset: DnCl supernatant

Input: `DnCl_supernatant_10ppm_min_peak_height_1e6_full_Feature_table.tsv`

P450 DnCl supernatant candidate screening (v6; two-route enzyme-vs-WT; 10 ppm, minimum peak height 1e6; 07252026)

This notebook replaces the previous z-score / DMSO-log2-threshold route.

For each enzyme, the three unlabeled replicates and the three D4 replicates are screened **independently**, each against all 36 DMSO samples. A feature passes a treatment screen only when:

1. raw intensity is greater than zero in all three treatment replicates;
2. `log2FC = log2((mean normalized raw treatment + 1) / (mean normalized raw DMSO + 1)) >= 3`;
3. normalized-raw treatment mean is greater than the DMSO mean;
4. two-sided Welch t-test on normalized `log2(x+1)` values has BH-FDR `q < 0.05`.

BH correction is performed separately for every enzyme × treatment type across all tested features. After both screens, only same-enzyme unlabeled/labeled feature pairs matching D1-D4, 10 ppm, and 5 seconds are retained.

## Load data and organize samples

In [ ]:
INPUT_FILE = PROJECT_DIR + 'DnCl_supernatant_10ppm_min_peak_height_1e6_full_Feature_table.tsv'

data_supernatant_DnCl = pd.read_csv(INPUT_FILE, sep='\t', index_col=0)
print(f'Feature table shape: {data_supernatant_DnCl.shape}')
data_supernatant_DnCl[['mz', 'rtime']].head()

In [ ]:
DMSO_samples = sorted(x for x in data_supernatant_DnCl.columns if '_DMSO_' in x)
Estrone_samples = sorted(x for x in data_supernatant_DnCl.columns if '_Estrone_' in x)
D4_Estrone_samples = sorted(x for x in data_supernatant_DnCl.columns if '_D4-Estrone_' in x)
Estradiol_samples = sorted(x for x in data_supernatant_DnCl.columns if '_Estradiol_' in x)
D4_Estradiol_samples = sorted(x for x in data_supernatant_DnCl.columns if '_D4-Estradiol_' in x)

expected_counts = {
    'DMSO': 36,
    'Estrone': 36,
    'D4-Estrone': 36,
    'Estradiol': 36,
    'D4-Estradiol': 36,
}
observed_counts = {
    'DMSO': len(DMSO_samples),
    'Estrone': len(Estrone_samples),
    'D4-Estrone': len(D4_Estrone_samples),
    'Estradiol': len(Estradiol_samples),
    'D4-Estradiol': len(D4_Estradiol_samples),
}
if observed_counts != expected_counts:
    raise ValueError(f'Unexpected sample counts: {observed_counts}')

for treatment_samples in [
    Estrone_samples, D4_Estrone_samples,
    Estradiol_samples, D4_Estradiol_samples,
]:
    for enzyme in ordered_enzyme_names:
        samples_for_enzyme(treatment_samples, enzyme)

observed_counts

In [ ]:
feature_dict = data_supernatant_DnCl[['mz', 'rtime']].to_dict('index')
feature_list = [
    {'id': feature_id, 'mz': values['mz'], 'rtime': values['rtime']}
    for feature_id, values in feature_dict.items()
]
print(f'{len(feature_list):,} LC-MS features available for D1-D4 matching.')

### Estrone

In [ ]:
estrone_all_samples = DMSO_samples + Estrone_samples + D4_Estrone_samples
normalized_raw, normalized_log2, normalization_scale = normalize_top500_then_log2(
    data_supernatant_DnCl[estrone_all_samples]
)

plot_normalization_qc(
    data_supernatant_DnCl[estrone_all_samples], normalized_raw,
    'Estrone',
)
normalization_scale.describe()

In [ ]:
estrone_unlabeled_selected = screen_all_enzymes(
    data_supernatant_DnCl, normalized_raw, normalized_log2,
    Estrone_samples, DMSO_samples, ordered_enzyme_names,
)
estrone_labeled_selected = screen_all_enzymes(
    data_supernatant_DnCl, normalized_raw, normalized_log2,
    D4_Estrone_samples, DMSO_samples, ordered_enzyme_names,
)

In [ ]:
estrone_screen_counts = summarize_screen_counts(
    estrone_unlabeled_selected, estrone_labeled_selected, ordered_enzyme_names,
)
estrone_screen_counts

In [ ]:
estrone_matches = match_selected_routes(
    estrone_unlabeled_selected, estrone_labeled_selected,
    ordered_enzyme_names, feature_dict, feature_list,
)
estrone_results = matches_to_table(
    estrone_matches, estrone_unlabeled_selected,
    estrone_labeled_selected, feature_dict,
)
estrone_output = OUTPUT_DIR + "01_v7_estrone_DnCl_supernatant_treatment_enriched_deuterium_matches.tsv"
export_selected_columns(
    estrone_results, H_D_MATCH_EXPORT_COLUMNS, estrone_output
)

print(f'Estrone matched rows: {len(estrone_results):,}')
print(f'Estrone unique unlabeled features: {estrone_results.unlabeled_feature.nunique() if not estrone_results.empty else 0:,}')
print(f'Estrone unique labeled features: {estrone_results.labeled_feature.nunique() if not estrone_results.empty else 0:,}')
print(estrone_output)

### Estradiol

In [ ]:
# Reuse these variable names so the large Estrone matrices can be released.
estradiol_all_samples = DMSO_samples + Estradiol_samples + D4_Estradiol_samples
normalized_raw, normalized_log2, normalization_scale = normalize_top500_then_log2(
    data_supernatant_DnCl[estradiol_all_samples]
)

plot_normalization_qc(
    data_supernatant_DnCl[estradiol_all_samples], normalized_raw,
    'Estradiol',
)
normalization_scale.describe()

In [ ]:
estradiol_unlabeled_selected = screen_all_enzymes(
    data_supernatant_DnCl, normalized_raw, normalized_log2,
    Estradiol_samples, DMSO_samples, ordered_enzyme_names,
)
estradiol_labeled_selected = screen_all_enzymes(
    data_supernatant_DnCl, normalized_raw, normalized_log2,
    D4_Estradiol_samples, DMSO_samples, ordered_enzyme_names,
)

In [ ]:
estradiol_screen_counts = summarize_screen_counts(
    estradiol_unlabeled_selected, estradiol_labeled_selected, ordered_enzyme_names,
)
estradiol_screen_counts

In [ ]:
estradiol_matches = match_selected_routes(
    estradiol_unlabeled_selected, estradiol_labeled_selected,
    ordered_enzyme_names, feature_dict, feature_list,
)
estradiol_results = matches_to_table(
    estradiol_matches, estradiol_unlabeled_selected,
    estradiol_labeled_selected, feature_dict,
)
estradiol_output = OUTPUT_DIR + "01_v7_estradiol_DnCl_supernatant_treatment_enriched_deuterium_matches.tsv"
export_selected_columns(
    estradiol_results, H_D_MATCH_EXPORT_COLUMNS, estradiol_output
)

print(f'Estradiol matched rows: {len(estradiol_results):,}')
print(f'Estradiol unique unlabeled features: {estradiol_results.unlabeled_feature.nunique() if not estradiol_results.empty else 0:,}')
print(f'Estradiol unique labeled features: {estradiol_results.labeled_feature.nunique() if not estradiol_results.empty else 0:,}')
print(estradiol_output)

## Enzyme-versus-WT specificity screen

Only isotope-confirmed unlabeled feature candidates enter this stage. Detection
and the fixed 1e6 signal threshold use original raw intensities. Fold change
uses top-500-normalized raw means, and Welch testing uses normalized
`log2(x+1)` values.

All passing routes require enzyme detection in all three replicates:

1. **WT 0/3:** no t-test; require all three enzyme original raw intensities
   >= 1e6.
2. **WT 3/3:** require all three enzyme original raw intensities >= 1e6.
   WT has no raw-intensity threshold beyond 3/3 detection. Then require enzyme
   normalized-raw mean > WT normalized-raw mean, log2FC >= 1, and two-sided
   Welch BH-FDR q < 0.05.

WT 1/3 and WT 2/3 cases are labeled `ambiguous_sparse_WT` and cannot pass
enzyme specificity.


## Estrone: isotope-confirmed unlabeled features versus WT

In [ ]:
normalized_raw, normalized_log2, _ = normalize_top500_then_log2(
    data_supernatant_DnCl[estrone_all_samples]
)
estrone_specificity_stats = enzyme_vs_wt_candidate_statistics(
    estrone_results, data_supernatant_DnCl, normalized_raw, normalized_log2,
    Estrone_samples, ordered_enzyme_names,
)
estrone_specific_pairs = retain_specific_isotope_pairs(
    estrone_results, estrone_specificity_stats
)
estrone_specificity_summary = specificity_summary(estrone_specificity_stats)
estrone_specificity_summary

In [ ]:
estrone_stats_output = OUTPUT_DIR + "02_v7_estrone_DnCl_supernatant_unlabeled_features_enzyme_vs_WT_statistics.tsv"
estrone_specific_output = OUTPUT_DIR + "03_v7_estrone_DnCl_supernatant_enzyme_specific_isotope_pairs.tsv"
export_selected_columns(
    estrone_specificity_stats, ENZYME_VS_WT_EXPORT_COLUMNS,
    estrone_stats_output,
)
export_selected_columns(
    estrone_specific_pairs, ENZYME_SPECIFIC_PAIR_EXPORT_COLUMNS,
    estrone_specific_output,
)

estrone_unique_unlabeled_features = (
    estrone_specific_pairs.loc[
        :, UNIQUE_ENZYME_SPECIFIC_FEATURE_EXPORT_COLUMNS
    ]
    .drop_duplicates()
    .sort_values(['enzyme', 'unlabeled_feature'], kind='stable')
    .reset_index(drop=True)
)
estrone_unique_unlabeled_output = OUTPUT_DIR + "04_v7_estrone_DnCl_supernatant_unique_enzyme_specific_unlabeled_features.tsv"

export_selected_columns(
    estrone_unique_unlabeled_features,
    UNIQUE_ENZYME_SPECIFIC_FEATURE_EXPORT_COLUMNS,
    estrone_unique_unlabeled_output,
)
print(f'Estrone tested enzyme-unlabeled-feature combinations: {len(estrone_specificity_stats):,}')
print(f'Estrone passing enzyme-unlabeled-feature combinations: {estrone_specificity_stats.pass_enzyme_vs_wt.sum():,}')
print(f'Estrone passing isotope-pair rows: {len(estrone_specific_pairs):,}')
print(estrone_stats_output)
print(estrone_specific_output)
print(
    f'Estrone unique enzyme-feature rows: '
    f'{len(estrone_unique_unlabeled_features):,}'
)
print(estrone_unique_unlabeled_output)

## Estradiol: isotope-confirmed unlabeled features versus WT

In [ ]:
normalized_raw, normalized_log2, _ = normalize_top500_then_log2(
    data_supernatant_DnCl[estradiol_all_samples]
)
estradiol_specificity_stats = enzyme_vs_wt_candidate_statistics(
    estradiol_results, data_supernatant_DnCl, normalized_raw, normalized_log2,
    Estradiol_samples, ordered_enzyme_names,
)
estradiol_specific_pairs = retain_specific_isotope_pairs(
    estradiol_results, estradiol_specificity_stats
)
estradiol_specificity_summary = specificity_summary(estradiol_specificity_stats)
estradiol_specificity_summary

In [ ]:
estradiol_stats_output = OUTPUT_DIR + "02_v7_estradiol_DnCl_supernatant_unlabeled_features_enzyme_vs_WT_statistics.tsv"
estradiol_specific_output = OUTPUT_DIR + "03_v7_estradiol_DnCl_supernatant_enzyme_specific_isotope_pairs.tsv"
export_selected_columns(
    estradiol_specificity_stats, ENZYME_VS_WT_EXPORT_COLUMNS,
    estradiol_stats_output,
)
export_selected_columns(
    estradiol_specific_pairs, ENZYME_SPECIFIC_PAIR_EXPORT_COLUMNS,
    estradiol_specific_output,
)

estradiol_unique_unlabeled_features = (
    estradiol_specific_pairs.loc[
        :, UNIQUE_ENZYME_SPECIFIC_FEATURE_EXPORT_COLUMNS
    ]
    .drop_duplicates()
    .sort_values(['enzyme', 'unlabeled_feature'], kind='stable')
    .reset_index(drop=True)
)
estradiol_unique_unlabeled_output = OUTPUT_DIR + "04_v7_estradiol_DnCl_supernatant_unique_enzyme_specific_unlabeled_features.tsv"

export_selected_columns(
    estradiol_unique_unlabeled_features,
    UNIQUE_ENZYME_SPECIFIC_FEATURE_EXPORT_COLUMNS,
    estradiol_unique_unlabeled_output,
)
print(f'Estradiol tested enzyme-unlabeled-feature combinations: {len(estradiol_specificity_stats):,}')
print(f'Estradiol passing enzyme-unlabeled-feature combinations: {estradiol_specificity_stats.pass_enzyme_vs_wt.sum():,}')
print(f'Estradiol passing isotope-pair rows: {len(estradiol_specific_pairs):,}')
print(estradiol_stats_output)
print(estradiol_specific_output)
print(
    f'Estradiol unique enzyme-feature rows: '
    f'{len(estradiol_unique_unlabeled_features):,}'
)
print(estradiol_unique_unlabeled_output)

## Estrone candidate features heatmap


In [ ]:
_, normalized_log2, _ = normalize_top500_then_log2(
    data_supernatant_DnCl[estrone_all_samples]
)
plot_v6_candidate_features_heatmap(
    normalized_log2,
    Estrone_samples,
    estrone_specificity_stats,
    ordered_enzyme_names,
    'Estrone',
    'supernatant',
    OUTPUT_DIR + "v7_Estrone_candidate_features_supernatant_row_zscore_heatmap.pdf",
    OUTPUT_DIR + "05_v7_estrone_DnCl_supernatant_all_WT_passing_unlabeled_features_row_zscore_matrix.tsv"
)


## Estradiol candidate features heatmap


In [ ]:
_, normalized_log2, _ = normalize_top500_then_log2(
    data_supernatant_DnCl[estradiol_all_samples]
)
plot_v6_candidate_features_heatmap(
    normalized_log2,
    Estradiol_samples,
    estradiol_specificity_stats,
    ordered_enzyme_names,
    'Estradiol',
    'supernatant',
    OUTPUT_DIR + "v7_Estradiol_candidate_features_supernatant_row_zscore_heatmap.pdf",
    OUTPUT_DIR + "05_v7_estradiol_DnCl_supernatant_all_WT_passing_unlabeled_features_row_zscore_matrix.tsv"
)


**Summary**

Stage 1 is unchanged from v5: independently enriched unlabeled features and
labeled features are matched within the same enzyme at D1-D4, 10 ppm, and
5 seconds.

The v6 enzyme-versus-WT stage has only two passing routes. WT 0/3 uses the
original-raw detection-limit rule without a t-test. WT 3/3 uses normalized-raw
direction/log2FC and a Welch test on normalized `log2(x+1)` values with BH-FDR.
WT 1/3 and WT 2/3 are always retained as `ambiguous_sparse_WT` diagnostics and
never pass specificity.

Passing unlabeled features are merged back to isotope matches. The candidate overview heatmap contains all v6 WT-passing unlabeled features and uses the final v9 heatmap layout. Heatmap row z-scores are visualization-only and do not affect selection.


# v9_b Step 1: Dataset: DnHz pellets

Input: `DnHz_pellets_10ppm_min_peak_height_1e6_full_Feature_table.tsv`

P450 DnHz pellets candidate screening (v6; two-route enzyme-vs-WT; 10 ppm, minimum peak height 1e6; 07252026)

This notebook applies the complete treatment-enrichment, isotope-matching,
enzyme-versus-WT, and specificity-heatmap workflow to the 10 ppm DnHz
pellet feature table.

Each enzyme's three unlabeled replicates and three isotope-labeled replicates
are screened independently against all 36 DMSO samples. A feature passes when:

1. raw intensity is greater than zero in all three treatment replicates;
2. `log2FC >= 3` using normalized-raw means with a `+1` pseudocount;
3. treatment mean is greater than the DMSO mean;
4. two-sided Welch testing on normalized `log2(x+1)` values has BH-FDR
   `q < 0.05`.

The experimentally observed isotope ranges are used:

- D4-Pregnenolone: D1 only;
- D9-Progesterone: D1-D5;
- D6-DHEA: D1-D6.

Only independently enriched unlabeled/labeled feature pairs from the same enzyme,
within 10 ppm and 5 seconds, are retained.

In [ ]:
SUBSTRATE_CONFIGS = {
    'Pregnenolone': {
        'slug': 'pregnenolone',
        'labeled_name': 'D4-Pregnenolone',
        'max_observed_deuterium': 1,
    },
    'Progesterone': {
        'slug': 'progesterone',
        'labeled_name': 'D9-Progesterone',
        'max_observed_deuterium': 5,
    },
    'DHEA': {
        'slug': 'dhea',
        'labeled_name': 'D6-DHEA',
        'max_observed_deuterium': 6,
    },
}

In [ ]:
H_D_MATCH_EXPORT_COLUMNS

In [ ]:
MATCH_COLUMNS = [
    'unlabeled_feature', 'unlabeled_feature_mz', 'unlabeled_feature_rtime', 'enzyme',
    'unlabeled_detection_count', 'unlabeled_treatment_mean_normalized_raw',
    'unlabeled_dmso_mean_normalized_raw', 'unlabeled_log2FC',
    'unlabeled_welch_p', 'unlabeled_fdr_q',
    'labeled_feature', 'labeled_feature_mz', 'labeled_feature_rtime',
    'deuterium_label', 'labeled_detection_count',
    'labeled_treatment_mean_normalized_raw',
    'labeled_dmso_mean_normalized_raw', 'labeled_log2FC',
    'labeled_welch_p', 'labeled_fdr_q',
]


In [ ]:
MATCH_COLUMNS == H_D_MATCH_EXPORT_COLUMNS

## Load data and organize the three substrate pairs


In [ ]:
INPUT_FILE = PROJECT_DIR + 'DnHz_pellets_10ppm_min_peak_height_1e6_full_Feature_table.tsv'

data_pellet_DnHz = pd.read_csv(INPUT_FILE, sep='\t', index_col=0)
print(f'Feature table shape: {data_pellet_DnHz.shape}')
INPUT_FILE

In [ ]:
DMSO_samples = sorted(
    x for x in data_pellet_DnHz.columns if '_DMSO_' in x
)

for substrate, config in SUBSTRATE_CONFIGS.items():
    config['unlabeled_samples'] = sorted(
        x for x in data_pellet_DnHz.columns
        if f'_{substrate}_' in x
    )
    config['labeled_samples'] = sorted(
        x for x in data_pellet_DnHz.columns
        if f"_{config['labeled_name']}_" in x
    )

observed_counts = {'DMSO': len(DMSO_samples)}
for substrate, config in SUBSTRATE_CONFIGS.items():
    observed_counts[substrate] = len(config['unlabeled_samples'])
    observed_counts[config['labeled_name']] = len(config['labeled_samples'])

expected_counts = {name: 36 for name in observed_counts}
if observed_counts != expected_counts:
    raise ValueError(f'Unexpected sample counts: {observed_counts}')

for config in SUBSTRATE_CONFIGS.values():
    for treatment_samples in [
        config['unlabeled_samples'], config['labeled_samples']
    ]:
        for enzyme in ordered_enzyme_names:
            samples_for_enzyme(treatment_samples, enzyme)

observed_counts


In [ ]:
feature_dict = data_pellet_DnHz[['mz', 'rtime']].to_dict('index')
feature_list = [
    {'id': feature_id, 'mz': values['mz'], 'rtime': values['rtime']}
    for feature_id, values in feature_dict.items()
]
print(f'{len(feature_list):,} LC-MS features available for isotope matching.')
print({
    substrate: [
        label for _, label in deuterium_search_range(config['max_observed_deuterium'])
    ]
    for substrate, config in SUBSTRATE_CONFIGS.items()
})


### Pregnenolone

In [ ]:
pregnenolone_config = SUBSTRATE_CONFIGS['Pregnenolone']
pregnenolone_unlabeled_samples = pregnenolone_config['unlabeled_samples']
pregnenolone_labeled_samples = pregnenolone_config['labeled_samples']
pregnenolone_all_samples = (
    DMSO_samples + pregnenolone_unlabeled_samples + pregnenolone_labeled_samples
)
normalized_raw, normalized_log2, normalization_scale = (
    normalize_top500_then_log2(data_pellet_DnHz[pregnenolone_all_samples])
)
plot_normalization_qc(
    data_pellet_DnHz[pregnenolone_all_samples], normalized_raw, 'Pregnenolone',
)
normalization_scale.describe()


In [ ]:
pregnenolone_unlabeled_selected = screen_all_enzymes(
    data_pellet_DnHz, normalized_raw, normalized_log2,
    pregnenolone_unlabeled_samples, DMSO_samples, ordered_enzyme_names,
)
pregnenolone_labeled_selected = screen_all_enzymes(
    data_pellet_DnHz, normalized_raw, normalized_log2,
    pregnenolone_labeled_samples, DMSO_samples, ordered_enzyme_names,
)

In [ ]:
pregnenolone_screen_counts = summarize_screen_counts(
    pregnenolone_unlabeled_selected, pregnenolone_labeled_selected, ordered_enzyme_names,
)
pregnenolone_screen_counts

In [ ]:
pregnenolone_matches = match_selected_routes(
    pregnenolone_unlabeled_selected, pregnenolone_labeled_selected,
    ordered_enzyme_names, feature_dict, feature_list,
    deuterium_search_range(pregnenolone_config['max_observed_deuterium']),
)
pregnenolone_results = matches_to_table(
    pregnenolone_matches, pregnenolone_unlabeled_selected,
    pregnenolone_labeled_selected, feature_dict,
)
pregnenolone_output = (
    OUTPUT_DIR + "01_v7_pregnenolone_DnHz_pellets_treatment_enriched_deuterium_matches.tsv"
)
export_selected_columns(
    pregnenolone_results, H_D_MATCH_EXPORT_COLUMNS, pregnenolone_output
)

retained_labels = (
    sorted(pregnenolone_results['deuterium_label'].unique())
    if not pregnenolone_results.empty else []
)
print(f'Pregnenolone isotope range retained: {retained_labels}')
print(f'Pregnenolone matched rows: {len(pregnenolone_results):,}')
print(
    f'Pregnenolone unique unlabeled features: '
    f'{pregnenolone_results.unlabeled_feature.nunique() if not pregnenolone_results.empty else 0:,}'
)
print(
    f'Pregnenolone unique labeled features: '
    f'{pregnenolone_results.labeled_feature.nunique() if not pregnenolone_results.empty else 0:,}'
)
print(pregnenolone_output)


### Progesterone

In [ ]:
progesterone_config = SUBSTRATE_CONFIGS['Progesterone']
progesterone_unlabeled_samples = progesterone_config['unlabeled_samples']
progesterone_labeled_samples = progesterone_config['labeled_samples']
progesterone_all_samples = (
    DMSO_samples + progesterone_unlabeled_samples + progesterone_labeled_samples
)
normalized_raw, normalized_log2, normalization_scale = (
    normalize_top500_then_log2(data_pellet_DnHz[progesterone_all_samples])
)
plot_normalization_qc(
    data_pellet_DnHz[progesterone_all_samples], normalized_raw, 'Progesterone',
)
normalization_scale.describe()


In [ ]:
progesterone_unlabeled_selected = screen_all_enzymes(
    data_pellet_DnHz, normalized_raw, normalized_log2,
    progesterone_unlabeled_samples, DMSO_samples, ordered_enzyme_names,
)
progesterone_labeled_selected = screen_all_enzymes(
    data_pellet_DnHz, normalized_raw, normalized_log2,
    progesterone_labeled_samples, DMSO_samples, ordered_enzyme_names,
)


In [ ]:
progesterone_screen_counts = summarize_screen_counts(
    progesterone_unlabeled_selected, progesterone_labeled_selected, ordered_enzyme_names,
)
progesterone_screen_counts


In [ ]:
progesterone_matches = match_selected_routes(
    progesterone_unlabeled_selected, progesterone_labeled_selected,
    ordered_enzyme_names, feature_dict, feature_list,
    deuterium_search_range(progesterone_config['max_observed_deuterium']),
)
progesterone_results = matches_to_table(
    progesterone_matches, progesterone_unlabeled_selected,
    progesterone_labeled_selected, feature_dict,
)
progesterone_output = OUTPUT_DIR + "01_v7_progesterone_DnHz_pellets_treatment_enriched_deuterium_matches.tsv"

export_selected_columns(
    progesterone_results, H_D_MATCH_EXPORT_COLUMNS, progesterone_output
)

retained_labels = (
    sorted(progesterone_results['deuterium_label'].unique())
    if not progesterone_results.empty else []
)
print(f'Progesterone isotope range retained: {retained_labels}')
print(f'Progesterone matched rows: {len(progesterone_results):,}')
print(
    f'Progesterone unique unlabeled features: '
    f'{progesterone_results.unlabeled_feature.nunique() if not progesterone_results.empty else 0:,}'
)
print(
    f'Progesterone unique labeled features: '
    f'{progesterone_results.labeled_feature.nunique() if not progesterone_results.empty else 0:,}'
)
print(progesterone_output)


### DHEA


In [ ]:
dhea_config = SUBSTRATE_CONFIGS['DHEA']
dhea_unlabeled_samples = dhea_config['unlabeled_samples']
dhea_labeled_samples = dhea_config['labeled_samples']
dhea_all_samples = (
    DMSO_samples + dhea_unlabeled_samples + dhea_labeled_samples
)
normalized_raw, normalized_log2, normalization_scale = (
    normalize_top500_then_log2(data_pellet_DnHz[dhea_all_samples])
)
plot_normalization_qc(
    data_pellet_DnHz[dhea_all_samples], normalized_raw, 'DHEA',
)
normalization_scale.describe()


In [ ]:
dhea_unlabeled_selected = screen_all_enzymes(
    data_pellet_DnHz, normalized_raw, normalized_log2,
    dhea_unlabeled_samples, DMSO_samples, ordered_enzyme_names,
)
dhea_labeled_selected = screen_all_enzymes(
    data_pellet_DnHz, normalized_raw, normalized_log2,
    dhea_labeled_samples, DMSO_samples, ordered_enzyme_names,
)

In [ ]:
dhea_screen_counts = summarize_screen_counts(
    dhea_unlabeled_selected, dhea_labeled_selected, ordered_enzyme_names,
)
dhea_screen_counts

In [ ]:
dhea_matches = match_selected_routes(
    dhea_unlabeled_selected, dhea_labeled_selected,
    ordered_enzyme_names, feature_dict, feature_list,
    deuterium_search_range(dhea_config['max_observed_deuterium']),
)
dhea_results = matches_to_table(
    dhea_matches, dhea_unlabeled_selected,
    dhea_labeled_selected, feature_dict,
)
dhea_output = OUTPUT_DIR + "01_v7_dhea_DnHz_pellets_treatment_enriched_deuterium_matches.tsv"

export_selected_columns(
    dhea_results, H_D_MATCH_EXPORT_COLUMNS, dhea_output
)

retained_labels = (
    sorted(dhea_results['deuterium_label'].unique())
    if not dhea_results.empty else []
)
print(f'DHEA isotope range retained: {retained_labels}')
print(f'DHEA matched rows: {len(dhea_results):,}')
print(
    f'DHEA unique unlabeled features: '
    f'{dhea_results.unlabeled_feature.nunique() if not dhea_results.empty else 0:,}'
)
print(
    f'DHEA unique labeled features: '
    f'{dhea_results.labeled_feature.nunique() if not dhea_results.empty else 0:,}'
)
print(dhea_output)


## Enzyme-versus-WT specificity screen

Only isotope-confirmed unlabeled feature candidates enter this stage. Detection
and the fixed 1e6 signal threshold use original raw intensities. Fold change
uses top-500-normalized raw means, and Welch testing uses normalized
`log2(x+1)` values.

All passing routes require enzyme detection in all three replicates:

1. **WT 0/3:** no t-test; require all three enzyme original raw intensities
   >= 1e6.
2. **WT 3/3:** require all three enzyme original raw intensities >= 1e6.
   WT has no raw-intensity threshold beyond 3/3 detection. Then require enzyme
   normalized-raw mean > WT normalized-raw mean, log2FC >= 1, and two-sided
   Welch BH-FDR q < 0.05.

WT 1/3 and WT 2/3 cases are labeled `ambiguous_sparse_WT` and cannot pass
enzyme specificity.


## Pregnenolone: isotope-confirmed unlabeled features versus WT


In [ ]:
normalized_raw, normalized_log2, _ = (
    normalize_top500_then_log2(data_pellet_DnHz[pregnenolone_all_samples])
)
pregnenolone_specificity_stats = enzyme_vs_wt_candidate_statistics(
    pregnenolone_results, data_pellet_DnHz,
    normalized_raw, normalized_log2,
    pregnenolone_unlabeled_samples, ordered_enzyme_names,
)
pregnenolone_specific_pairs = retain_specific_isotope_pairs(
    pregnenolone_results, pregnenolone_specificity_stats
)
pregnenolone_specificity_summary = specificity_summary(
    pregnenolone_specificity_stats
)
pregnenolone_specificity_summary


In [ ]:
pregnenolone_stats_output = OUTPUT_DIR + "02_v7_pregnenolone_DnHz_pellets_unlabeled_features_enzyme_vs_WT_statistics.tsv"
pregnenolone_specific_output = OUTPUT_DIR + "03_v7_pregnenolone_DnHz_pellets_enzyme_specific_isotope_pairs.tsv"

export_selected_columns(
    pregnenolone_specificity_stats, ENZYME_VS_WT_EXPORT_COLUMNS,
    pregnenolone_stats_output,
)
export_selected_columns(
    pregnenolone_specific_pairs, ENZYME_SPECIFIC_PAIR_EXPORT_COLUMNS,
    pregnenolone_specific_output,
)

pregnenolone_unique_unlabeled_features = (
    pregnenolone_specific_pairs.loc[
        :, UNIQUE_ENZYME_SPECIFIC_FEATURE_EXPORT_COLUMNS
    ]
    .drop_duplicates()
    .sort_values(['enzyme', 'unlabeled_feature'], kind='stable')
    .reset_index(drop=True)
)
pregnenolone_unique_unlabeled_output = OUTPUT_DIR + "04_v7_pregnenolone_DnHz_pellets_unique_enzyme_specific_unlabeled_features.tsv"

export_selected_columns(
    pregnenolone_unique_unlabeled_features,
    UNIQUE_ENZYME_SPECIFIC_FEATURE_EXPORT_COLUMNS,
    pregnenolone_unique_unlabeled_output,
)

print(
    f'Pregnenolone tested enzyme-unlabeled-feature combinations: '
    f'{len(pregnenolone_specificity_stats):,}'
)
print(
    f'Pregnenolone passing enzyme-unlabeled-feature combinations: '
    f'{pregnenolone_specificity_stats.pass_enzyme_vs_wt.sum():,}'
)
print(
    f'Pregnenolone passing isotope-pair rows: '
    f'{len(pregnenolone_specific_pairs):,}'
)
print(pregnenolone_stats_output)
print(pregnenolone_specific_output)
print(
    f'Pregnenolone unique enzyme-feature rows: '
    f'{len(pregnenolone_unique_unlabeled_features):,}'
)
print(pregnenolone_unique_unlabeled_output)


## Progesterone: isotope-confirmed unlabeled features versus WT


In [ ]:
normalized_raw, normalized_log2, _ = (
    normalize_top500_then_log2(data_pellet_DnHz[progesterone_all_samples])
)
progesterone_specificity_stats = enzyme_vs_wt_candidate_statistics(
    progesterone_results, data_pellet_DnHz,
    normalized_raw, normalized_log2,
    progesterone_unlabeled_samples, ordered_enzyme_names,
)
progesterone_specific_pairs = retain_specific_isotope_pairs(
    progesterone_results, progesterone_specificity_stats
)
progesterone_specificity_summary = specificity_summary(
    progesterone_specificity_stats
)
progesterone_specificity_summary


In [ ]:
progesterone_stats_output = OUTPUT_DIR + "02_v7_progesterone_DnHz_pellets_unlabeled_features_enzyme_vs_WT_statistics.tsv"
progesterone_specific_output = OUTPUT_DIR + "03_v7_progesterone_DnHz_pellets_enzyme_specific_isotope_pairs.tsv"

export_selected_columns(
    progesterone_specificity_stats, ENZYME_VS_WT_EXPORT_COLUMNS,
    progesterone_stats_output,
)
export_selected_columns(
    progesterone_specific_pairs, ENZYME_SPECIFIC_PAIR_EXPORT_COLUMNS,
    progesterone_specific_output,
)

progesterone_unique_unlabeled_features = (
    progesterone_specific_pairs.loc[
        :, UNIQUE_ENZYME_SPECIFIC_FEATURE_EXPORT_COLUMNS
    ]
    .drop_duplicates()
    .sort_values(['enzyme', 'unlabeled_feature'], kind='stable')
    .reset_index(drop=True)
)
progesterone_unique_unlabeled_output = OUTPUT_DIR + "04_v7_progesterone_DnHz_pellets_unique_enzyme_specific_unlabeled_features.tsv"

export_selected_columns(
    progesterone_unique_unlabeled_features,
    UNIQUE_ENZYME_SPECIFIC_FEATURE_EXPORT_COLUMNS,
    progesterone_unique_unlabeled_output,
)

print(
    f'Progesterone tested enzyme-unlabeled-feature combinations: '
    f'{len(progesterone_specificity_stats):,}'
)
print(
    f'Progesterone passing enzyme-unlabeled-feature combinations: '
    f'{progesterone_specificity_stats.pass_enzyme_vs_wt.sum():,}'
)
print(
    f'Progesterone passing isotope-pair rows: '
    f'{len(progesterone_specific_pairs):,}'
)
print(progesterone_stats_output)
print(progesterone_specific_output)
print(
    f'Progesterone unique enzyme-feature rows: '
    f'{len(progesterone_unique_unlabeled_features):,}'
)
print(progesterone_unique_unlabeled_output)


## DHEA: isotope-confirmed unlabeled features versus WT


In [ ]:
normalized_raw, normalized_log2, _ = (
    normalize_top500_then_log2(data_pellet_DnHz[dhea_all_samples])
)
dhea_specificity_stats = enzyme_vs_wt_candidate_statistics(
    dhea_results, data_pellet_DnHz,
    normalized_raw, normalized_log2,
    dhea_unlabeled_samples, ordered_enzyme_names,
)
dhea_specific_pairs = retain_specific_isotope_pairs(
    dhea_results, dhea_specificity_stats
)
dhea_specificity_summary = specificity_summary(
    dhea_specificity_stats
)
dhea_specificity_summary


In [ ]:
dhea_stats_output = OUTPUT_DIR + "02_v7_dhea_DnHz_pellets_unlabeled_features_enzyme_vs_WT_statistics.tsv"
dhea_specific_output = OUTPUT_DIR + "03_v7_dhea_DnHz_pellets_enzyme_specific_isotope_pairs.tsv"

export_selected_columns(
    dhea_specificity_stats, ENZYME_VS_WT_EXPORT_COLUMNS,
    dhea_stats_output,
)
export_selected_columns(
    dhea_specific_pairs, ENZYME_SPECIFIC_PAIR_EXPORT_COLUMNS,
    dhea_specific_output,
)

dhea_unique_unlabeled_features = (
    dhea_specific_pairs.loc[
        :, UNIQUE_ENZYME_SPECIFIC_FEATURE_EXPORT_COLUMNS
    ]
    .drop_duplicates()
    .sort_values(['enzyme', 'unlabeled_feature'], kind='stable')
    .reset_index(drop=True)
)
dhea_unique_unlabeled_output = OUTPUT_DIR + "04_v7_dhea_DnHz_pellets_unique_enzyme_specific_unlabeled_features.tsv"

export_selected_columns(
    dhea_unique_unlabeled_features,
    UNIQUE_ENZYME_SPECIFIC_FEATURE_EXPORT_COLUMNS,
    dhea_unique_unlabeled_output,
)

print(
    f'DHEA tested enzyme-unlabeled-feature combinations: '
    f'{len(dhea_specificity_stats):,}'
)
print(
    f'DHEA passing enzyme-unlabeled-feature combinations: '
    f'{dhea_specificity_stats.pass_enzyme_vs_wt.sum():,}'
)
print(
    f'DHEA passing isotope-pair rows: '
    f'{len(dhea_specific_pairs):,}'
)
print(dhea_stats_output)
print(dhea_specific_output)
print(
    f'Dhea unique enzyme-feature rows: '
    f'{len(dhea_unique_unlabeled_features):,}'
)
print(dhea_unique_unlabeled_output)


## Pregnenolone candidate features heatmap


In [ ]:
_, normalized_log2, _ = normalize_top500_then_log2(
    data_pellet_DnHz[pregnenolone_all_samples]
)
plot_v6_candidate_features_heatmap(
    normalized_log2,
    pregnenolone_unlabeled_samples,
    pregnenolone_specificity_stats,
    ordered_enzyme_names,
    'Pregnenolone',
    'pellets',
    OUTPUT_DIR + "v7_Pregnenolone_candidate_features_pellets_row_zscore_heatmap.pdf",
    OUTPUT_DIR + "05_v7_pregnenolone_DnHz_pellets_all_WT_passing_unlabeled_features_row_zscore_matrix.tsv"
)


## Progesterone candidate features heatmap


In [ ]:
_, normalized_log2, _ = normalize_top500_then_log2(
    data_pellet_DnHz[progesterone_all_samples]
)
plot_v6_candidate_features_heatmap(
    normalized_log2,
    progesterone_unlabeled_samples,
    progesterone_specificity_stats,
    ordered_enzyme_names,
    'Progesterone',
    'pellets',
    OUTPUT_DIR + "v7_Progesterone_candidate_features_pellets_row_zscore_heatmap.pdf",
    OUTPUT_DIR + "05_v7_progesterone_DnHz_pellets_all_WT_passing_unlabeled_features_row_zscore_matrix.tsv"
)


## DHEA candidate features heatmap


In [ ]:
_, normalized_log2, _ = normalize_top500_then_log2(
    data_pellet_DnHz[dhea_all_samples]
)
plot_v6_candidate_features_heatmap(
    normalized_log2,
    dhea_unlabeled_samples,
    dhea_specificity_stats,
    ordered_enzyme_names,
    'DHEA',
    'pellets',
    OUTPUT_DIR + "v7_DHEA_candidate_features_pellets_row_zscore_heatmap.pdf",
    OUTPUT_DIR + "05_v7_dhea_DnHz_pellets_all_WT_passing_unlabeled_features_row_zscore_matrix.tsv"
)


**Summary**

Stage 1 independently screens each unlabeled and isotope-labeled route against
all DMSO samples, then applies the experimentally observed D1, D1-D5, or D1-D6
matching rule within the same enzyme at 10 ppm and 5 seconds.

The v6 enzyme-versus-WT stage has only two passing routes. WT 0/3 uses the
original-raw detection-limit rule without a t-test. WT 3/3 uses normalized-raw
direction/log2FC and a Welch test on normalized `log2(x+1)` values with BH-FDR.
WT 1/3 and WT 2/3 are labeled `ambiguous_sparse_WT` and never pass specificity.

Passing unlabeled features are merged back to isotope matches. The candidate overview heatmap contains all v6 WT-passing unlabeled features and uses the final v9 heatmap layout. Heatmap row z-scores are visualization-only and do not affect selection.


# v9_b Step 1: Dataset: DnHz supernatant

Input: `DnHz_supernatant_10ppm_min_peak_height_1e6_full_Feature_table.tsv`

P450 DnHz supernatant candidate screening (v6; two-route enzyme-vs-WT; 10 ppm, minimum peak height 1e6; 07252026)

This notebook applies the complete treatment-enrichment, isotope-matching,
enzyme-versus-WT, and specificity-heatmap workflow to the 10 ppm DnHz
supernatant feature table.

Each enzyme's three unlabeled replicates and three isotope-labeled replicates
are screened independently against all 36 DMSO samples. A feature passes when:

1. raw intensity is greater than zero in all three treatment replicates;
2. `log2FC >= 3` using normalized-raw means with a `+1` pseudocount;
3. treatment mean is greater than the DMSO mean;
4. two-sided Welch testing on normalized `log2(x+1)` values has BH-FDR
   `q < 0.05`.

The experimentally observed isotope ranges are used:

- D4-Pregnenolone: D1 only;
- D9-Progesterone: D1-D5;
- D6-DHEA: D1-D6.

Only independently enriched unlabeled/labeled feature pairs from the same enzyme,
within 10 ppm and 5 seconds, are retained.

In [ ]:
INPUT_FILE = PROJECT_DIR + 'DnHz_supernatant_10ppm_min_peak_height_1e6_full_Feature_table.tsv'

data_supernatant_DnHz = pd.read_csv(INPUT_FILE, sep='\t', index_col=0)
print(f'Feature table shape: {data_supernatant_DnHz.shape}')

In [ ]:
DMSO_samples = sorted(
    x for x in data_supernatant_DnHz.columns if '_DMSO_' in x
)

for substrate, config in SUBSTRATE_CONFIGS.items():
    config['unlabeled_samples'] = sorted(
        x for x in data_supernatant_DnHz.columns
        if f'_{substrate}_' in x
    )
    config['labeled_samples'] = sorted(
        x for x in data_supernatant_DnHz.columns
        if f"_{config['labeled_name']}_" in x
    )

observed_counts = {'DMSO': len(DMSO_samples)}
for substrate, config in SUBSTRATE_CONFIGS.items():
    observed_counts[substrate] = len(config['unlabeled_samples'])
    observed_counts[config['labeled_name']] = len(config['labeled_samples'])

expected_counts = {name: 36 for name in observed_counts}
if observed_counts != expected_counts:
    raise ValueError(f'Unexpected sample counts: {observed_counts}')

for config in SUBSTRATE_CONFIGS.values():
    for treatment_samples in [
        config['unlabeled_samples'], config['labeled_samples']
    ]:
        for enzyme in ordered_enzyme_names:
            samples_for_enzyme(treatment_samples, enzyme)

observed_counts


In [ ]:
feature_dict = data_supernatant_DnHz[['mz', 'rtime']].to_dict('index')
feature_list = [
    {'id': feature_id, 'mz': values['mz'], 'rtime': values['rtime']}
    for feature_id, values in feature_dict.items()
]
print(f'{len(feature_list):,} LC-MS features available for isotope matching.')
print({
    substrate: [
        label for _, label in deuterium_search_range(config['max_observed_deuterium'])
    ]
    for substrate, config in SUBSTRATE_CONFIGS.items()
})


### Pregnenolone

In [ ]:
pregnenolone_config = SUBSTRATE_CONFIGS['Pregnenolone']
pregnenolone_unlabeled_samples = pregnenolone_config['unlabeled_samples']
pregnenolone_labeled_samples = pregnenolone_config['labeled_samples']
pregnenolone_all_samples = (
    DMSO_samples + pregnenolone_unlabeled_samples + pregnenolone_labeled_samples
)
normalized_raw, normalized_log2, normalization_scale = (
    normalize_top500_then_log2(data_supernatant_DnHz[pregnenolone_all_samples])
)
plot_normalization_qc(
    data_supernatant_DnHz[pregnenolone_all_samples], normalized_raw, 'Pregnenolone',
)
normalization_scale.describe()


In [ ]:
pregnenolone_unlabeled_selected = screen_all_enzymes(
    data_supernatant_DnHz, normalized_raw, normalized_log2,
    pregnenolone_unlabeled_samples, DMSO_samples, ordered_enzyme_names,
)
pregnenolone_labeled_selected = screen_all_enzymes(
    data_supernatant_DnHz, normalized_raw, normalized_log2,
    pregnenolone_labeled_samples, DMSO_samples, ordered_enzyme_names,
)

In [ ]:
pregnenolone_screen_counts = summarize_screen_counts(
    pregnenolone_unlabeled_selected, pregnenolone_labeled_selected, ordered_enzyme_names,
)
pregnenolone_screen_counts

In [ ]:
pregnenolone_matches = match_selected_routes(
    pregnenolone_unlabeled_selected, pregnenolone_labeled_selected,
    ordered_enzyme_names, feature_dict, feature_list,
    deuterium_search_range(pregnenolone_config['max_observed_deuterium']),
)
pregnenolone_results = matches_to_table(
    pregnenolone_matches, pregnenolone_unlabeled_selected,
    pregnenolone_labeled_selected, feature_dict,
)
pregnenolone_output = OUTPUT_DIR + "01_v7_pregnenolone_DnHz_supernatant_treatment_enriched_deuterium_matches.tsv"

export_selected_columns(
    pregnenolone_results, H_D_MATCH_EXPORT_COLUMNS, pregnenolone_output
)

retained_labels = (
    sorted(pregnenolone_results['deuterium_label'].unique())
    if not pregnenolone_results.empty else []
)
print(f'Pregnenolone isotope range retained: {retained_labels}')
print(f'Pregnenolone matched rows: {len(pregnenolone_results):,}')
print(
    f'Pregnenolone unique unlabeled features: '
    f'{pregnenolone_results.unlabeled_feature.nunique() if not pregnenolone_results.empty else 0:,}'
)
print(
    f'Pregnenolone unique labeled features: '
    f'{pregnenolone_results.labeled_feature.nunique() if not pregnenolone_results.empty else 0:,}'
)
print(pregnenolone_output)


### Progesterone


In [ ]:
progesterone_config = SUBSTRATE_CONFIGS['Progesterone']
progesterone_unlabeled_samples = progesterone_config['unlabeled_samples']
progesterone_labeled_samples = progesterone_config['labeled_samples']
progesterone_all_samples = (
    DMSO_samples + progesterone_unlabeled_samples + progesterone_labeled_samples
)
normalized_raw, normalized_log2, normalization_scale = (
    normalize_top500_then_log2(data_supernatant_DnHz[progesterone_all_samples])
)
plot_normalization_qc(
    data_supernatant_DnHz[progesterone_all_samples], normalized_raw, 'Progesterone',
)
normalization_scale.describe()


In [ ]:
progesterone_unlabeled_selected = screen_all_enzymes(
    data_supernatant_DnHz, normalized_raw, normalized_log2,
    progesterone_unlabeled_samples, DMSO_samples, ordered_enzyme_names,
)
progesterone_labeled_selected = screen_all_enzymes(
    data_supernatant_DnHz, normalized_raw, normalized_log2,
    progesterone_labeled_samples, DMSO_samples, ordered_enzyme_names,
)

In [ ]:
progesterone_screen_counts = summarize_screen_counts(
    progesterone_unlabeled_selected, progesterone_labeled_selected, ordered_enzyme_names,
)
progesterone_screen_counts

In [ ]:
progesterone_matches = match_selected_routes(
    progesterone_unlabeled_selected, progesterone_labeled_selected,
    ordered_enzyme_names, feature_dict, feature_list,
    deuterium_search_range(progesterone_config['max_observed_deuterium']),
)
progesterone_results = matches_to_table(
    progesterone_matches, progesterone_unlabeled_selected,
    progesterone_labeled_selected, feature_dict,
)
progesterone_output = OUTPUT_DIR + "01_v7_progesterone_DnHz_supernatant_treatment_enriched_deuterium_matches.tsv"

export_selected_columns(
    progesterone_results, H_D_MATCH_EXPORT_COLUMNS, progesterone_output
)

retained_labels = (
    sorted(progesterone_results['deuterium_label'].unique())
    if not progesterone_results.empty else []
)
print(f'Progesterone isotope range retained: {retained_labels}')
print(f'Progesterone matched rows: {len(progesterone_results):,}')
print(
    f'Progesterone unique unlabeled features: '
    f'{progesterone_results.unlabeled_feature.nunique() if not progesterone_results.empty else 0:,}'
)
print(
    f'Progesterone unique labeled features: '
    f'{progesterone_results.labeled_feature.nunique() if not progesterone_results.empty else 0:,}'
)
print(progesterone_output)


### DHEA

In [ ]:
dhea_config = SUBSTRATE_CONFIGS['DHEA']
dhea_unlabeled_samples = dhea_config['unlabeled_samples']
dhea_labeled_samples = dhea_config['labeled_samples']
dhea_all_samples = (
    DMSO_samples + dhea_unlabeled_samples + dhea_labeled_samples
)
normalized_raw, normalized_log2, normalization_scale = (
    normalize_top500_then_log2(data_supernatant_DnHz[dhea_all_samples])
)
plot_normalization_qc(
    data_supernatant_DnHz[dhea_all_samples], normalized_raw, 'DHEA',
)
normalization_scale.describe()


In [ ]:
dhea_unlabeled_selected = screen_all_enzymes(
    data_supernatant_DnHz, normalized_raw, normalized_log2,
    dhea_unlabeled_samples, DMSO_samples, ordered_enzyme_names,
)
dhea_labeled_selected = screen_all_enzymes(
    data_supernatant_DnHz, normalized_raw, normalized_log2,
    dhea_labeled_samples, DMSO_samples, ordered_enzyme_names,
)

In [ ]:
dhea_screen_counts = summarize_screen_counts(
    dhea_unlabeled_selected, dhea_labeled_selected, ordered_enzyme_names,
)
dhea_screen_counts

In [ ]:
dhea_matches = match_selected_routes(
    dhea_unlabeled_selected, dhea_labeled_selected,
    ordered_enzyme_names, feature_dict, feature_list,
    deuterium_search_range(dhea_config['max_observed_deuterium']),
)
dhea_results = matches_to_table(
    dhea_matches, dhea_unlabeled_selected,
    dhea_labeled_selected, feature_dict,
)
dhea_output = OUTPUT_DIR + "01_v7_dhea_DnHz_supernatant_treatment_enriched_deuterium_matches.tsv"

export_selected_columns(
    dhea_results, H_D_MATCH_EXPORT_COLUMNS, dhea_output
)

retained_labels = (
    sorted(dhea_results['deuterium_label'].unique())
    if not dhea_results.empty else []
)
print(f'DHEA isotope range retained: {retained_labels}')
print(f'DHEA matched rows: {len(dhea_results):,}')
print(
    f'DHEA unique unlabeled features: '
    f'{dhea_results.unlabeled_feature.nunique() if not dhea_results.empty else 0:,}'
)
print(
    f'DHEA unique labeled features: '
    f'{dhea_results.labeled_feature.nunique() if not dhea_results.empty else 0:,}'
)
print(dhea_output)


## Enzyme-versus-WT specificity screen

Only isotope-confirmed unlabeled feature candidates enter this stage. Detection
and the fixed 1e6 signal threshold use original raw intensities. Fold change
uses top-500-normalized raw means, and Welch testing uses normalized
`log2(x+1)` values.

All passing routes require enzyme detection in all three replicates:

1. **WT 0/3:** no t-test; require all three enzyme original raw intensities
   >= 1e6.
2. **WT 3/3:** require all three enzyme original raw intensities >= 1e6.
   WT has no raw-intensity threshold beyond 3/3 detection. Then require enzyme
   normalized-raw mean > WT normalized-raw mean, log2FC >= 1, and two-sided
   Welch BH-FDR q < 0.05.

WT 1/3 and WT 2/3 cases are labeled `ambiguous_sparse_WT` and cannot pass
enzyme specificity.


## Pregnenolone: isotope-confirmed unlabeled features versus WT


In [ ]:
normalized_raw, normalized_log2, _ = (
    normalize_top500_then_log2(data_supernatant_DnHz[pregnenolone_all_samples])
)
pregnenolone_specificity_stats = enzyme_vs_wt_candidate_statistics(
    pregnenolone_results, data_supernatant_DnHz,
    normalized_raw, normalized_log2,
    pregnenolone_unlabeled_samples, ordered_enzyme_names,
)
pregnenolone_specific_pairs = retain_specific_isotope_pairs(
    pregnenolone_results, pregnenolone_specificity_stats
)
pregnenolone_specificity_summary = specificity_summary(
    pregnenolone_specificity_stats
)
pregnenolone_specificity_summary


In [ ]:
pregnenolone_stats_output = OUTPUT_DIR + "02_v7_pregnenolone_DnHz_supernatant_unlabeled_features_enzyme_vs_WT_statistics.tsv"
pregnenolone_specific_output = OUTPUT_DIR + "03_v7_pregnenolone_DnHz_supernatant_enzyme_specific_isotope_pairs.tsv"

export_selected_columns(
    pregnenolone_specificity_stats, ENZYME_VS_WT_EXPORT_COLUMNS,
    pregnenolone_stats_output,
)
export_selected_columns(
    pregnenolone_specific_pairs, ENZYME_SPECIFIC_PAIR_EXPORT_COLUMNS,
    pregnenolone_specific_output,
)

pregnenolone_unique_unlabeled_features = (
    pregnenolone_specific_pairs.loc[
        :, UNIQUE_ENZYME_SPECIFIC_FEATURE_EXPORT_COLUMNS
    ]
    .drop_duplicates()
    .sort_values(['enzyme', 'unlabeled_feature'], kind='stable')
    .reset_index(drop=True)
)
pregnenolone_unique_unlabeled_output = OUTPUT_DIR + "04_v7_pregnenolone_DnHz_supernatant_unique_enzyme_specific_unlabeled_features.tsv"

export_selected_columns(
    pregnenolone_unique_unlabeled_features,
    UNIQUE_ENZYME_SPECIFIC_FEATURE_EXPORT_COLUMNS,
    pregnenolone_unique_unlabeled_output,
)

print(
    f'Pregnenolone tested enzyme-unlabeled-feature combinations: '
    f'{len(pregnenolone_specificity_stats):,}'
)
print(
    f'Pregnenolone passing enzyme-unlabeled-feature combinations: '
    f'{pregnenolone_specificity_stats.pass_enzyme_vs_wt.sum():,}'
)
print(
    f'Pregnenolone passing isotope-pair rows: '
    f'{len(pregnenolone_specific_pairs):,}'
)
print(pregnenolone_stats_output)
print(pregnenolone_specific_output)
print(
    f'Pregnenolone unique enzyme-feature rows: '
    f'{len(pregnenolone_unique_unlabeled_features):,}'
)
print(pregnenolone_unique_unlabeled_output)


## Progesterone: isotope-confirmed unlabeled features versus WT


In [ ]:
normalized_raw, normalized_log2, _ = (
    normalize_top500_then_log2(data_supernatant_DnHz[progesterone_all_samples])
)
progesterone_specificity_stats = enzyme_vs_wt_candidate_statistics(
    progesterone_results, data_supernatant_DnHz,
    normalized_raw, normalized_log2,
    progesterone_unlabeled_samples, ordered_enzyme_names,
)
progesterone_specific_pairs = retain_specific_isotope_pairs(
    progesterone_results, progesterone_specificity_stats
)
progesterone_specificity_summary = specificity_summary(
    progesterone_specificity_stats
)
progesterone_specificity_summary

In [ ]:
progesterone_stats_output = OUTPUT_DIR + "02_v7_progesterone_DnHz_supernatant_unlabeled_features_enzyme_vs_WT_statistics.tsv"
progesterone_specific_output = OUTPUT_DIR + "03_v7_progesterone_DnHz_supernatant_enzyme_specific_isotope_pairs.tsv"

export_selected_columns(
    progesterone_specificity_stats, ENZYME_VS_WT_EXPORT_COLUMNS,
    progesterone_stats_output,
)
export_selected_columns(
    progesterone_specific_pairs, ENZYME_SPECIFIC_PAIR_EXPORT_COLUMNS,
    progesterone_specific_output,
)

progesterone_unique_unlabeled_features = (
    progesterone_specific_pairs.loc[
        :, UNIQUE_ENZYME_SPECIFIC_FEATURE_EXPORT_COLUMNS
    ]
    .drop_duplicates()
    .sort_values(['enzyme', 'unlabeled_feature'], kind='stable')
    .reset_index(drop=True)
)
progesterone_unique_unlabeled_output = OUTPUT_DIR + "04_v7_progesterone_DnHz_supernatant_unique_enzyme_specific_unlabeled_features.tsv"

export_selected_columns(
    progesterone_unique_unlabeled_features,
    UNIQUE_ENZYME_SPECIFIC_FEATURE_EXPORT_COLUMNS,
    progesterone_unique_unlabeled_output,
)

print(
    f'Progesterone tested enzyme-unlabeled-feature combinations: '
    f'{len(progesterone_specificity_stats):,}'
)
print(
    f'Progesterone passing enzyme-unlabeled-feature combinations: '
    f'{progesterone_specificity_stats.pass_enzyme_vs_wt.sum():,}'
)
print(
    f'Progesterone passing isotope-pair rows: '
    f'{len(progesterone_specific_pairs):,}'
)
print(progesterone_stats_output)
print(progesterone_specific_output)
print(
    f'Progesterone unique enzyme-feature rows: '
    f'{len(progesterone_unique_unlabeled_features):,}'
)
print(progesterone_unique_unlabeled_output)

### DHEA: isotope-confirmed unlabeled features versus WT

In [ ]:
normalized_raw, normalized_log2, _ = (
    normalize_top500_then_log2(data_supernatant_DnHz[dhea_all_samples])
)
dhea_specificity_stats = enzyme_vs_wt_candidate_statistics(
    dhea_results, data_supernatant_DnHz,
    normalized_raw, normalized_log2,
    dhea_unlabeled_samples, ordered_enzyme_names,
)
dhea_specific_pairs = retain_specific_isotope_pairs(
    dhea_results, dhea_specificity_stats
)
dhea_specificity_summary = specificity_summary(
    dhea_specificity_stats
)
dhea_specificity_summary


In [ ]:
dhea_stats_output = OUTPUT_DIR + "02_v7_dhea_DnHz_supernatant_unlabeled_features_enzyme_vs_WT_statistics.tsv"
dhea_specific_output = OUTPUT_DIR + "03_v7_dhea_DnHz_supernatant_enzyme_specific_isotope_pairs.tsv"

export_selected_columns(
    dhea_specificity_stats, ENZYME_VS_WT_EXPORT_COLUMNS,
    dhea_stats_output,
)
export_selected_columns(
    dhea_specific_pairs, ENZYME_SPECIFIC_PAIR_EXPORT_COLUMNS,
    dhea_specific_output,
)

dhea_unique_unlabeled_features = (
    dhea_specific_pairs.loc[
        :, UNIQUE_ENZYME_SPECIFIC_FEATURE_EXPORT_COLUMNS
    ]
    .drop_duplicates()
    .sort_values(['enzyme', 'unlabeled_feature'], kind='stable')
    .reset_index(drop=True)
)
dhea_unique_unlabeled_output = OUTPUT_DIR + "04_v7_dhea_DnHz_supernatant_unique_enzyme_specific_unlabeled_features.tsv"

export_selected_columns(
    dhea_unique_unlabeled_features,
    UNIQUE_ENZYME_SPECIFIC_FEATURE_EXPORT_COLUMNS,
    dhea_unique_unlabeled_output,
)

print(
    f'DHEA tested enzyme-unlabeled-feature combinations: '
    f'{len(dhea_specificity_stats):,}'
)
print(
    f'DHEA passing enzyme-unlabeled-feature combinations: '
    f'{dhea_specificity_stats.pass_enzyme_vs_wt.sum():,}'
)
print(
    f'DHEA passing isotope-pair rows: '
    f'{len(dhea_specific_pairs):,}'
)
print(dhea_stats_output)
print(dhea_specific_output)
print(
    f'Dhea unique enzyme-feature rows: '
    f'{len(dhea_unique_unlabeled_features):,}'
)
print(dhea_unique_unlabeled_output)


## candidate feature heatmaps

Each heatmap contains all v6 WT-passing unlabeled features, including single-enzyme and shared candidates. The three normalized `log2(x+1)` replicates are averaged within each of the 11 enzymes and WT, followed by a row z-score across the 12 group means. The v9 final heatmap layout is used for direct visual comparison; this visualization does not change candidate selection or statistical testing.


## Pregnenolone candidate features heatmap


In [ ]:
_, normalized_log2, _ = normalize_top500_then_log2(
    data_supernatant_DnHz[pregnenolone_all_samples]
)
plot_v6_candidate_features_heatmap(
    normalized_log2,
    pregnenolone_unlabeled_samples,
    pregnenolone_specificity_stats,
    ordered_enzyme_names,
    'Pregnenolone',
    'supernatant',
    OUTPUT_DIR + "v7_Pregnenolone_candidate_features_supernatant_row_zscore_heatmap.pdf",
    OUTPUT_DIR + "05_v7_pregnenolone_DnHz_supernatant_all_WT_passing_unlabeled_features_row_zscore_matrix.tsv"
)


## Progesterone candidate features heatmap


In [ ]:
_, normalized_log2, _ = normalize_top500_then_log2(
    data_supernatant_DnHz[progesterone_all_samples]
)
plot_v6_candidate_features_heatmap(
    normalized_log2,
    progesterone_unlabeled_samples,
    progesterone_specificity_stats,
    ordered_enzyme_names,
    'Progesterone',
    'supernatant',
    OUTPUT_DIR + "v7_Progesterone_candidate_features_supernatant_row_zscore_heatmap.pdf",
    OUTPUT_DIR + "05_v7_progesterone_DnHz_supernatant_all_WT_passing_unlabeled_features_row_zscore_matrix.tsv"
)


## DHEA candidate features heatmap


In [ ]:
_, normalized_log2, _ = normalize_top500_then_log2(
    data_supernatant_DnHz[dhea_all_samples]
)
plot_v6_candidate_features_heatmap(
    normalized_log2,
    dhea_unlabeled_samples,
    dhea_specificity_stats,
    ordered_enzyme_names,
    'DHEA',
    'supernatant',
    OUTPUT_DIR + "v7_DHEA_candidate_features_supernatant_row_zscore_heatmap.pdf",
    OUTPUT_DIR + "05_v7_dhea_DnHz_supernatant_all_WT_passing_unlabeled_features_row_zscore_matrix.tsv"
)


**Summary**

Stage 1 independently screens each unlabeled and isotope-labeled route against
all DMSO samples, then applies the experimentally observed D1, D1-D5, or D1-D6
matching rule within the same enzyme at 10 ppm and 5 seconds.

The v6 enzyme-versus-WT stage has only two passing routes. WT 0/3 uses the
original-raw detection-limit rule without a t-test. WT 3/3 uses normalized-raw
direction/log2FC and a Welch test on normalized `log2(x+1)` values with BH-FDR.
WT 1/3 and WT 2/3 are labeled `ambiguous_sparse_WT` and never pass specificity.

Passing unlabeled features are merged back to isotope matches. The candidate overview heatmap contains all v6 WT-passing unlabeled features and uses the final v9 heatmap layout. Heatmap row z-scores are visualization-only and do not affect selection.


Final output checks

In [ ]:
tsv_files = [f for f in os.listdir(OUTPUT_DIR) if f.endswith('.tsv')]
pdf_files = [f for f in os.listdir(OUTPUT_DIR) if f.endswith('.pdf')]
print(f'TSV tables: {len(tsv_files)}')
print(f'PDF heatmaps: {len(pdf_files)}')